# 🏺 Eng–Akk Low Resource Sent Alignment 🔍📜

> **Project Goal:** Create high-quality parallel training data for Neural Machine Translation (NMT) of complex, low-resource ancient languages.

## 🧐 The Challenge: "Broken Clay, Complex Tongues"
Aligning Akkadian cuneiform with English translations poses unique challenges that break standard alignment tools:
*   **Physical Damage:** Tablets are often broken, containing gaps (`[...]`) and missing context (lacunae).
*   **Morphological Gap:** Akkadian is a morphologically rich Semitic language with root-based derivation and complex case systems. A single Akkadian word often maps to an entire English phrase, and word order varies significantly from English.
*   **Data Scarcity:** We lack the millions of sentence pairs used in modern NMT.
*   **Technological Gap:** Unlike high-resource languages, Akkadian lacks pre-trained embedding models (like BERT/RoBERTa). This makes modern alignment approaches like **SimAlign**—which rely on semantic similarity in vector space—impossible to use. We are forced to innovate with classical statistical methods.
*   **Script Complexity:** The writing system mixes syllabic phonemes with logograms (Sumerograms), making direct lexical matching difficult.

## 💡 The Solution: Normalized Regret & Monotonic Anchoring
Standard algorithms (like Gale-Church) fail here because of the **Density Problem**. Semitic languages like Akkadian are significantly more dense than English—a single Akkadian word often translates to an entire English clause. This violates the length-ratio assumptions of Gale-Church. Furthermore, standard methods let "easy" sentences hoard tokens, leaving damaged or complex sentences abandoned (assigned NULL).

This notebook implements a **custom Unsupervised Pipeline** featuring:
1.  **Normalized Regret Scoring (Novelty!)**: A new objective function that judges alignments by their *potential*. It prevents the model from abandoning hard-to-align sentences.
    *   *Formula*: $\text{Satisfaction} = \frac{\text{Actual Score}}{\text{Ideal Potential}}$
    
2.  **Smart Entity Resolution**: 
    *   Uses a **Lexicon** to identify Proper Nouns (PN), Divine Names (DN), and Geographic Names (GN).
    *   Applies **Soundex** phonetic matching to bridge the gap between Akkadian transliteration and English spellings (e.g., aligning *Aššur* with *Ashur*).

3.  **Monotonic Anchoring**:
    *   High-confidence matches (Entities, Numbers, Sumerograms) become **anchors** in the Dynamic Programming matrix.
    *   The Viterbi algorithm enforces a **monotonic alignment path**, constraining the search space to ensure chronological consistency between source and target.

## 🛠️ Pipeline Architecture
1.  **Preprocessing**: Normalization of Sumerograms, determinatives, and gaps.
2.  **Lexical Training**: Unsupervised learning of word translations via IBM Model 1.
3.  **Scout Phase**: Calculating the theoretical "alignment ceiling" for every sentence.
4.  **Global Optimization**: A Dynamic Programming pass that maximizes global *satisfaction* while respecting monotonic constraints.

---
*Ready to resurrect Babylon? Let's dive in.* 🧱➡️📄

## Pipeline Architecture Summary

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                    AKKADIAN-ENGLISH SENTENCE ALIGNER                        │
│                         End-to-End Pipeline                                 │
└─────────────────────────────────────────────────────────────────────────────┘

╔═════════════════════════════════════════════════════════════════════════════╗
║  INPUT DATA                                                                 ║
╠═════════════════════════════════════════════════════════════════════════════╣
║  • train.csv (document-level parallel texts for alignment testing)          ║
║  • Sentences_Oare_FirstWord_LinNum.csv (sentence-level for model training)  ║
║  • OA_Lexicon_eBL.csv (form → lexeme mapping, word types: PN, GN, DN)       ║
║  • eBL_Dictionary.csv (dictionary priors for translation)                   ║
╚═════════════════════════════════════════════════════════════════════════════╝
                                    │
        ┌───────────────────────────┴───────────────────────────┐
        ▼                                                       ▼
╔═══════════════════════════════════╗   ╔═══════════════════════════════════╗
║  PART 1: MODEL TRAINING (§1.1-1.9)║   ║  EXTERNAL RESOURCES (§1.3)        ║
╠═══════════════════════════════════╣   ╠═══════════════════════════════════╣
║                                   ║   ║  • Lexicon: form → lexeme         ║
║  §1.2 Text Normalization          ║   ║  • Word Types: PN, GN, DN, etc.   ║
║    • Akkadian vowel normalization ║   ║  • Dictionary translation priors  ║
║    • Gap marker standardization   ║   ║  • Sumerogram → English mappings  ║
║      (x x x → <big_gap>)          ║   ╚═══════════════════════════════════╝
║      (... → <big_gap>)            ║
║      (x → <gap>)                  ║
║                                   ║
║  §1.4-1.5 Dataset & Splits        ║
║    • Load parallel corpus         ║
║    • Train/Dev/Test split (80/10) ║
║                                   ║
║  §1.6 IBM Model 1 Training        ║
║    ┌─────────────┐ ┌─────────────┐║
║    │  Forward    │ │  Reverse    │║
║    │  Eng → Akk  │ │  Akk → Eng  │║
║    │  P(a|e)     │ │  P(e|a)     │║
║    └─────────────┘ └─────────────┘║
║    + Phonetic matching (Soundex)  ║
║    + Type constraints (PN↔PN)     ║
║    + Dictionary priors            ║
║                                   ║
║  §1.7 Length Model                ║
║    • Log-normal ratio prior       ║
║    • μ, σ from training data      ║
║                                   ║
║  §1.9 Evaluation                  ║
║    • Boundary accuracy (±1, ±3)   ║
╚═══════════════════════════════════╝
                │
                ▼  (lexical_model.json, length_model.json)
╔═══════════════════════════════════════════════════════════════════════════╗
║  PART 2: GLOBAL ALIGNMENT PIPELINE (§2.1-2.5)                             ║
╠═══════════════════════════════════════════════════════════════════════════╣
║                                                                           ║
║  §2.1 Text Cleaning                                                       ║
║    • Gap marker normalization via replace_gaps()                          ║
║    • Heuristic sets: pronouns, particles, clause starters                 ║
║    • English sentence splitting (abbreviation handling)                   ║
║                                                                           ║
║  §2.2 AlignmentScorer ─────────────────────────────────────────────────── ║
║  ┌─────────────────────────────────────────────────────────────────────┐  ║
║  │  Multi-Feature Scoring Function                                     │  ║
║  │  ┌─────────────────┐  ┌─────────────────┐  ┌─────────────────────┐  │  ║
║  │  │ Bidirectional   │  │ IDF Weighting   │  │ Phonetic Match     │  │  ║
║  │  │ √(P(a|e)×P(e|a))│  │ log(N/df)       │  │ Soundex(PN)        │  │  ║
║  │  └─────────────────┘  └─────────────────┘  └─────────────────────┘  │  ║
║  │  ┌─────────────────┐  ┌─────────────────┐  ┌─────────────────────┐  │  ║
║  │  │ Length Prior    │  │ Fuzzy Match     │  │ Boundary Heuristic │  │  ║
║  │  │ log-normal(μ,σ) │  │ Edit distance   │  │ Clause starters    │  │  ║
║  │  └─────────────────┘  └─────────────────┘  └─────────────────────┘  │  ║
║  └─────────────────────────────────────────────────────────────────────┘  ║
║                                      │                                    ║
║                                      ▼                                    ║
║  §2.3 Global Alignment (Viterbi DP + Normalized Regret) ─────────────────  ║
║  ┌─────────────────────────────────────────────────────────────────────┐  ║
║  │  1. SCOUT PHASE (compute_ideal_scores)                              │  ║
║  │     • Find best possible score for each sentence                    │  ║
║  │     • Used to normalize scores relative to sentence potential       │  ║
║  │                                                                     │  ║
║  │  2. ANCHOR DETECTION                                                │  ║
║  │     • Proper nouns (Soundex matching via lexicon)                   │  ║
║  │     • Gap markers (<gap>, <big_gap>)                                │  ║
║  │     • Numbers and Sumerograms (DUMU→son, KÙ.BABBAR→silver)          │  ║
║  │                                                                     │  ║
║  │  3. DYNAMIC PROGRAMMING with NORMALIZED REGRET                      │  ║
║  │     • dp[i][j] = best satisfaction for Eng[0:i] → Akk[0:j]          │  ║
║  │     • satisfaction = actual_score / |ideal_score|                   │  ║
║  │     • Prevents easy sentences from abandoning hard ones             │  ║
║  │     • Diagonal prior (penalize deviation from expected position)    │  ║
║  │     • Clause-starter penalty for merging (if, when, whoever)        │  ║
║  │                                                                     │  ║
║  │  4. BACKTRACKING                                                    │  ║
║  │     • Extract optimal alignment path                                │  ║
║  │     • Build (English sentence → Akkadian span) mappings             │  ║
║  └─────────────────────────────────────────────────────────────────────┘  ║
║                                                                           ║
║  §2.4 Training Set Generation                                             ║
║    • Quality filtering (score threshold, length bounds)                   ║
║    • Partial alignment extraction for noisy documents                     ║
║                                                                           ║
╚═══════════════════════════════════════════════════════════════════════════╝
                │
                ▼
╔═══════════════════════════════════════════════════════════════════════════╗
║  OUTPUT                                                                   ║
╠═══════════════════════════════════════════════════════════════════════════╣
║  • Sentence-aligned pairs (JSONL)                                         ║
║  • Alignment scores and confidence metrics                                ║
║  • NULL alignment markers for untranslated segments                       ║
║  • Merge indicators for multi-sentence alignments                         ║
╚═══════════════════════════════════════════════════════════════════════════╝
```

**Key Algorithm: Normalized Regret Scoring**

The core innovation is **normalized regret** scoring that prevents "easy" sentences from dominating "hard" sentences in the global optimization:

1. **Scout Phase**: For each sentence, find its ideal score (best possible alignment ignoring other sentences)
2. **Satisfaction Ratio**: `satisfaction = actual_score / |ideal_score|`
3. **Result**: A hard sentence achieving its potential scores the same as an easy sentence achieving its potential

This eliminates the problem where sentences with many out-of-vocabulary words get abandoned (NULL) because their absolute scores are low even when correctly aligned.

**Notebook Section Reference:**
| Part | Sections | Description |
|------|----------|-------------|
| **1** | 1.1-1.12 | Model Training & Evaluation |
| **2** | 2.1-2.5 | Global Alignment Pipeline |

**Key Functions:**
| Function | Purpose |
|----------|---------|
| `replace_gaps()` | Normalize gap markers (x x x → `<big_gap>`, x → `<gap>`) |
| `compute_ideal_scores()` | Scout phase: find best possible score per sentence |
| `global_align()` | Viterbi DP with normalized regret scoring |
| `find_token_anchors()` | Detect proper nouns, Sumerograms, gaps as anchors |
| `AlignmentScorer.score_span()` | Multi-feature scoring (IDF, length, phonetics) |


---

## Part 1: Model Training & Evaluation

### 1.1 Imports and Configuration

In [ ]:
import csv
import re
import unicodedata
import collections
import json
import os
import math
import random
import statistics
import difflib
import pandas as pd
from collections import defaultdict
from functools import lru_cache

# Configuration - File Paths
TRAIN_FILE = '/kaggle/input/deep-past-initiative-machine-translation/train.csv'
SENTENCE_FILE = '/kaggle/input/deep-past-initiative-machine-translation/Sentences_Oare_FirstWord_LinNum.csv'
LEXICON_FILE = '/kaggle/input/deep-past-initiative-machine-translation/OA_Lexicon_eBL.csv'
DICTIONARY_FILE = '/kaggle/input/deep-past-initiative-machine-translation/eBL_Dictionary.csv'
ALIGNED_SEGMENTS_FILE = '/kaggle/input/manually-aligned-segments/aligned_segments.jsonl'

# Output Files
SPLITS_FILE = 'splits.json'
LEXICAL_MODEL_FILE = 'lexical_model.json'
LENGTH_MODEL_FILE = 'length_model.json'

# Model Hyperparameters
ITERATIONS = 15      # IBM Model 1 EM iterations
MIN_PROB = 0.001     # Minimum probability threshold for pruning
TOP_K = 50           # Top K translations to keep per word
RANDOM_SEED = 42     # For reproducibility

print("Configuration loaded.")

### 1.2 Text Processing Utilities

Functions for normalizing and tokenizing Akkadian and English text. Handles vowel length markers, special characters, and gap markers.

In [ ]:
@lru_cache(maxsize=10000)
def normalize_akkadian_string(s):
    """Normalize Akkadian string: remove vowel length markers but keep consonants (š, ṣ, ṭ)."""
    if not s: return ""
    s = unicodedata.normalize('NFC', s)
    
    # Map accented vowels to base vowels
    replacements = str.maketrans({
        'ā': 'a', 'ē': 'e', 'ī': 'i', 'ū': 'u',
        'â': 'a', 'ê': 'e', 'î': 'i', 'û': 'u',
        'á': 'a', 'é': 'e', 'í': 'i', 'ú': 'u',
        'à': 'a', 'è': 'e', 'ì': 'i', 'ù': 'u',
        'Ā': 'A', 'Ē': 'E', 'Ī': 'I', 'Ū': 'U',
        'Â': 'A', 'Ê': 'E', 'Î': 'I', 'Û': 'U',
        'Á': 'A', 'É': 'E', 'Í': 'I', 'Ú': 'U',
        'À': 'A', 'È': 'E', 'Ì': 'I', 'Ù': 'U'
    })
    return s.translate(replacements)

def tokenize_akkadian_for_indexing(text):
    """Tokenize Akkadian text by whitespace to match 'first_word_number' indexing."""
    if not text: return []
    return text.split()

def tokenize_akkadian(text):
    """
    Tokenize Akkadian text for alignment (IBM Model 1).
    Handles <gap> and <big_gap> tags as special tokens.
    """
    if not text: return []
    
    tokens = []
    raw_tokens = text.split() if '<' not in text else [t for t in re.findall(r'<big_gap>|<gap>|\S+', text) if t]
    
    for t in raw_tokens:
        if t in ['-', '.', '...']:
            continue
        if t.startswith('<') and t.endswith('>'):
            tokens.append(t)
        else:
            tokens.append(normalize_akkadian_string(t))
    return tokens

@lru_cache(maxsize=10000)
def tokenize_english(text):
    """English tokenization: lowercase, keep alphanumerics with internal hyphens/dots."""
    if not text: return []
    text = text.lower()
    text = re.sub(r'[()\[\]]', '', text)
    
    tokens = [t for t in re.findall(r'<big_gap>|<gap>|\w+(?:[\.\-]\w+)*', text, re.UNICODE) if t]
    
    # Filter out Akkadian-looking noise
    filtered_tokens = [t for t in tokens if t not in ['-a', '-i', '-ma', '-u', '-am']]
    return filtered_tokens

def replace_gaps(text):
    """Replace various gap markers with standardized <gap> and <big_gap> tokens."""
    if pd.isna(text): 
        return text

    # Handle spaced dots
    text = re.sub(r'(?:\.\s){2,}\.', '<big_gap>', text)
    text = re.sub(r'\[\.{3}(?:\s+\.{3})+\]\s+\.{3}(?:\s+\.{3})+', '<big_gap>', text)
    text = re.sub(r'\[\.{3}(?:\s+\.{3})+\]', '<big_gap>', text)
    text = re.sub(r'\.{3}(?:\s+\.{3})+', '<big_gap>', text)
    text = re.sub(r'\[x\]', '<gap>', text)
    text = re.sub(r'\bx{2,}\b', '<big_gap>', text)
    text = re.sub(r'\bx(?:\s+x)+\b', '<big_gap>', text)
    text = re.sub(r'(?<!\w)x(?!\w)', '<gap>', text)
    text = re.sub(r'\[…\]', '<big_gap>', text)
    text = re.sub(r'\[\.\.\.\]', '<big_gap>', text)
    text = re.sub(r'…', '<big_gap>', text)
    text = re.sub(r'\.\.\.', '<big_gap>', text)
    
    # Clean up any trailing periods after gap markers (e.g., "<big_gap>.." -> "<big_gap>")
    text = re.sub(r'(<(?:big_)?gap>)\.+', r'\1', text)
    
    return text

print("Text processing utilities loaded.")

### 1.3 Lexicon and Dictionary Loading

Load external resources: OA Lexicon (form→lexeme mapping) and eBL Dictionary (translation priors).

In [ ]:
def load_lexicon():
    """Load OA_Lexicon_eBL.csv to map forms to lexemes."""
    print(f"Loading lexicon from {LEXICON_FILE}...")
    form_to_lexeme = {}
    try:
        with open(LEXICON_FILE, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                form = row.get('form', '').strip()
                lexeme = row.get('lexeme', '').strip()
                if form and lexeme:
                    form_to_lexeme[form] = lexeme
                    norm_form = normalize_akkadian_string(form)
                    if norm_form not in form_to_lexeme:
                        form_to_lexeme[norm_form] = lexeme
        print(f"  Loaded {len(form_to_lexeme)} form->lexeme mappings.")
    except FileNotFoundError:
        print(f"  Warning: {LEXICON_FILE} not found.")
    return form_to_lexeme

def load_lexicon_types():
    """Load word types from OA_Lexicon_eBL.csv (PN, DN, GN, etc.)"""
    print("Loading lexicon types...")
    types = {}
    if os.path.exists(LEXICON_FILE):
        with open(LEXICON_FILE, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                form = row.get('form', '').strip()
                typ = row.get('type', '').strip()
                if form and typ:
                    types[form] = typ
        print(f"  Loaded {len(types)} type annotations.")
    return types

def load_dictionary_priors(form_to_lexeme):
    """Load eBL_Dictionary.csv to get English glosses for Akkadian lexemes."""
    print(f"Loading dictionary priors from {DICTIONARY_FILE}...")
    priors = collections.defaultdict(set)
    try:
        with open(DICTIONARY_FILE, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                word = row.get('word', '').strip()
                definition = row.get('definition', '').strip()
                if word and definition:
                    eng_tokens = tokenize_english(definition)
                    for eng in eng_tokens:
                        if len(eng) > 2:
                            priors[word].add(eng)
        print(f"  Loaded priors for {len(priors)} lexemes.")
    except FileNotFoundError:
        print(f"  Warning: {DICTIONARY_FILE} not found.")
    return priors

def get_akkadian_representation(token, form_to_lexeme):
    """Convert a raw Akkadian token to its representation (Lexeme ID or Normalized Form)."""
    if token in form_to_lexeme:
        return form_to_lexeme[token]
    norm_token = normalize_akkadian_string(token)
    if norm_token in form_to_lexeme:
        return form_to_lexeme[norm_token]
    return norm_token

print("Lexicon utilities loaded.")

### 1.4 Dataset Loading

Load and align the parallel corpus from raw CSV files (OARE texts with sentence-level annotations).

In [ ]:
def load_dataset(min_coverage=0.5):
    """
    Load and align the dataset from TRAIN_FILE and SENTENCE_FILE.
    Returns a list of document objects with sentence-level alignments.
    
    Args:
        min_coverage: Minimum ratio of sentence-file words to train.csv words.
                     Documents below this threshold are filtered out.
    """
    print("Loading dataset...")
    
    # 1. Load Full Akkadian Text AND Full English Translation from train.csv
    full_texts = {}
    full_translations = {}
    try:
        with open(TRAIN_FILE, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                full_texts[row['oare_id']] = row['transliteration']
                full_translations[row['oare_id']] = row.get('translation', '')
        print(f"  Loaded {len(full_texts)} documents from {TRAIN_FILE}")
    except FileNotFoundError:
        print(f"  Error: {TRAIN_FILE} not found.")
        return []

    # 2. Load Sentences
    sentences_by_doc = collections.defaultdict(list)
    try:
        with open(SENTENCE_FILE, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                if row['text_uuid'] in full_texts:
                    sentences_by_doc[row['text_uuid']].append(row)
        print(f"  Loaded sentences for {len(sentences_by_doc)} documents from {SENTENCE_FILE}")
    except FileNotFoundError:
        print(f"  Error: {SENTENCE_FILE} not found.")
        return []

    documents = []
    skipped_low_coverage = 0
    
    for doc_id, sents in sentences_by_doc.items():
        try:
            sents.sort(key=lambda x: float(x['first_word_number']))
        except ValueError:
            continue
        
        # Check coverage: compare sentence-file words to train.csv full translation
        full_translation = full_translations.get(doc_id, '')
        full_trans_words = len(full_translation.split())
        sent_words = sum(len(s.get('translation', '').split()) for s in sents)
        
        if full_trans_words > 0:
            coverage = sent_words / full_trans_words
            if coverage < min_coverage:
                skipped_low_coverage += 1
                continue  # Skip this document - sentence annotations are incomplete
            
        full_akk = full_texts[doc_id]
        akk_words = tokenize_akkadian_for_indexing(full_akk)
        
        # Map word index to (start, end) char offset
        word_idx_to_span = {}
        search_pos = 0
        valid_doc = True
        
        for i, word in enumerate(akk_words):
            start = full_akk.find(word, search_pos)
            if start == -1:
                valid_doc = False
                break
            end = start + len(word)
            word_idx_to_span[i+1] = (start, end)  # 1-based index
            search_pos = end
            
        if not valid_doc:
            continue

        doc_parts = []
        full_eng_parts = []
        current_eng_char = 0
        
        for i, sent in enumerate(sents):
            try:
                start_word_idx = int(float(sent['first_word_number']))
                
                # Verify word alignment
                expected_spelling = sent.get('first_word_spelling', '').strip()
                if expected_spelling and start_word_idx <= len(akk_words):
                    actual_word = akk_words[start_word_idx - 1]
                    if expected_spelling.replace('-', '').lower() != actual_word.replace('-', '').lower():
                        valid_doc = False
                        break
                
                # Determine end word index
                if i < len(sents) - 1:
                    end_word_idx = int(float(sents[i+1]['first_word_number'])) - 1
                else:
                    end_word_idx = len(akk_words)
                
                if start_word_idx > len(akk_words):
                    valid_doc = False
                    break
                    
                if end_word_idx < start_word_idx:
                    end_word_idx = start_word_idx

                # Get spans
                akk_start_char = word_idx_to_span[start_word_idx][0]
                akk_end_char = word_idx_to_span[end_word_idx][1]
                akk_part_text = full_akk[akk_start_char:akk_end_char]
                
                eng_part_text = sent['translation'].strip()
                full_eng_parts.append(eng_part_text)
                
                eng_start_char = current_eng_char
                eng_end_char = current_eng_char + len(eng_part_text)
                current_eng_char = eng_end_char + 1
                
                doc_parts.append({
                    'eng_text': eng_part_text,
                    'akk_text': akk_part_text,
                    'eng_span': (eng_start_char, eng_end_char),
                    'akk_span': (akk_start_char, akk_end_char)
                })
                
            except (ValueError, KeyError):
                valid_doc = False
                break
        
        if valid_doc and doc_parts:
            documents.append({
                'id': doc_id,
                'akk_text': full_akk,
                'eng_text': " ".join(full_eng_parts),
                'parts': doc_parts
            })
    
    print(f"  Successfully loaded {len(documents)} documents with sentence alignments.")
    print(f"  Skipped {skipped_low_coverage} documents with <{min_coverage*100:.0f}% sentence coverage.")
    return documents

print("Dataset loading function defined.")

### 1.5 Data Splitting

Create reproducible train/dev/test splits for model training and evaluation.

In [ ]:
def create_splits(documents, train_ratio=0.8, dev_ratio=0.1):
    """Create train/dev/test splits from documents."""
    print("Creating data splits...")
    
    doc_ids = [doc['id'] for doc in documents]
    
    if not doc_ids:
        print("  No documents found.")
        return None

    random.seed(RANDOM_SEED)
    random.shuffle(doc_ids)
    
    total = len(doc_ids)
    n_train = int(total * train_ratio)
    n_dev = int(total * dev_ratio)
    
    splits = {
        "train": doc_ids[:n_train],
        "dev": doc_ids[n_train:n_train+n_dev],
        "test": doc_ids[n_train+n_dev:]
    }
    
    print(f"  Train: {len(splits['train'])}, Dev: {len(splits['dev'])}, Test: {len(splits['test'])}")
    
    # Save to file
    with open(SPLITS_FILE, 'w', encoding='utf-8') as f:
        json.dump(splits, f, indent=2)
    print(f"  Saved to {SPLITS_FILE}")
    
    return splits

print("Data splitting function defined.")

### 1.6 Lexical Model Training (IBM Model 1)

Train bidirectional translation models using EM algorithm with:
- Type constraints (proper nouns, logograms)
- Phonetic matching (Soundex)
- Dictionary priors from eBL

In [ ]:
@lru_cache(maxsize=10000)
def soundex(name):
    """Compute Soundex code for phonetic matching."""
    if not name: return ""
    name = name.upper()
    soundex_code = name[0]
    
    dictionary = {"B": "1", "F": "1", "P": "1", "V": "1",
                  "C": "2", "G": "2", "J": "2", "K": "2", "Q": "2", "S": "2", "X": "2", "Z": "2",
                  "D": "3", "T": "3", "L": "4", "M": "5", "N": "5", "R": "6"}
    
    for char in name[1:]:
        code = dictionary.get(char, ".")
        if code != "." and code != soundex_code[-1]:
            soundex_code += code
    return soundex_code[:4].ljust(4, "0")

def get_type_penalty(s, t, s_type, t_type, reverse=False):
    """
    Return a multiplier (0.0 to 1.0) for alignment constraints.
    Enforces gap constraints, number constraints, proper noun constraints, etc.
    """
    GAP_TOKENS = {'<gap>', '<big_gap>'}
    s_is_gap = s in GAP_TOKENS
    t_is_gap = t in GAP_TOKENS
    
    if s_is_gap and not t_is_gap: return 0.0
    if t_is_gap and not s_is_gap: return 0.0
    if s_is_gap and t_is_gap: return 1.0

    # Number constraints
    NUMBER_WORDS = {
        'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine', 'ten',
        'eleven', 'twelve', 'thirteen', 'hundred', 'thousand', 'half', 'third', 'quarter',
        'ištēn', 'išten', 'šina', 'šinā', 'šalāš', 'erbe', 'ḫamiš', 'ešer', 'me\'at', 'līm'
    }

    s_starts_digit = s[0].isdigit() if s else False
    t_starts_digit = t[0].isdigit() if t else False
    s_is_alpha = s.isalpha() if s else False
    t_is_alpha = t.isalpha() if t else False
    
    if s_starts_digit and t_is_alpha and t.lower() not in NUMBER_WORDS:
        return 0.001
    if t_starts_digit and s_is_alpha and s.lower() not in NUMBER_WORDS:
        return 0.001

    # Garbage filter
    s_garbage = 'x' in s.lower() or '...' in s
    t_garbage = 'x' in t.lower() or '...' in t
    if s_garbage != t_garbage:
        return 0.001

    # Proper noun constraint
    if t_type and t_type in ['PN', 'DN', 'RN', 'GN', 'SN']:
        is_eng_pn = s[0].isupper() and len(s) > 1
        if not is_eng_pn:
            return 0.1

    # Length ratio constraint
    len_s, len_t = len(s), len(t)
    if len_s > 0 and len_t > 0:
        ratio = len_s / len_t
        if (ratio > 4.0 or ratio < 0.25) and min(len_s, len_t) < 3:
            return 0.01

    # Stopword constraints - strict mappings
    STRICT_MAP = {
        'and': {'u', 'ù'}, 'in': {'ina', 'i-na'}, 'of': {'ša', 'sa'},
        'with': {'išti', 'itti'}, 'from': {'ištu'}, 'not': {'lā', 'la', 'ul'},
    }
    
    s_lower = s.lower()
    t_lower = t.lower()
    
    if t_lower == '<null>':
        return 1.0

    if s_lower in STRICT_MAP:
        if any(t_lower.startswith(a) for a in STRICT_MAP[s_lower]) or t_lower in STRICT_MAP[s_lower]:
            return 5.0
        else:
            return 0.01

    return 1.0

print("Type penalty function defined.")

In [ ]:
# Patterns that indicate editorial/scholarly content (not actual translations)
EDITORIAL_PATTERNS = [
    r'\bsee\s+\w+\s+\d+',          # "see Smith 1984"
    r'\bcf\.\s*\w+',                # "cf. CAD"
    r'\blit\.\s*["\']',             # 'lit. "..."'
    r'\bi\.e\.\s',                  # "i.e. ..."
    r'\be\.g\.\s',                  # "e.g. ..."
    r'\bnote\s*:',                  # "Note:"
    r'\[sic\]',                     # "[sic]"
    r'\[lit\.\]',                   # "[lit.]"
    r'^\s*\[.*\]\s*$',              # Lines that are entirely in brackets
    r'\(\s*\d+\s*\)',               # "(123)" line numbers
    r'\beditor',                    # "editor's note"
    r'\brestored\b',                # "restored reading"
    r'\bbroken\s+away',             # "broken away"
    r'\bunreadable\b',              # "unreadable"
]

def is_editorial_content(eng_text):
    """Check if English text contains editorial/scholarly annotations."""
    eng_lower = eng_text.lower()
    for pattern in EDITORIAL_PATTERNS:
        if re.search(pattern, eng_lower, re.IGNORECASE):
            return True
    return False

def clean_training_examples(examples, min_ratio=0.2, max_ratio=5.0, 
                           min_akk_tokens=1, min_eng_tokens=1,
                           filter_editorial=True, verbose=True):
    """
    Clean training examples by filtering out problematic pairs.
    
    Filters:
    1. Extreme length ratios (Akkadian/English token ratio)
    2. Too few tokens on either side
    3. Editorial/scholarly annotations (not actual translations)
    
    Args:
        examples: List of training examples
        min_ratio: Minimum akk/eng token ratio (default: 0.2)
        max_ratio: Maximum akk/eng token ratio (default: 5.0)
        min_akk_tokens: Minimum Akkadian tokens (excluding <null>)
        min_eng_tokens: Minimum English tokens (excluding <null>)
        filter_editorial: Remove editorial content
        verbose: Print statistics
    
    Returns:
        Cleaned list of examples
    """
    if verbose:
        print(f"\n  Cleaning training data...")
        print(f"    Filters: ratio [{min_ratio:.1f}, {max_ratio:.1f}], min_tokens=({min_akk_tokens}, {min_eng_tokens})")
    
    cleaned = []
    stats = {
        'total': len(examples),
        'kept': 0,
        'filtered_ratio_low': 0,
        'filtered_ratio_high': 0,
        'filtered_too_short': 0,
        'filtered_editorial': 0,
    }
    
    for ex in examples:
        # Count tokens (excluding <null>)
        akk_count = len([t for t in ex['akk_tokens'] if t != '<null>'])
        eng_count = len([t for t in ex['eng_tokens'] if t != '<null>'])
        
        # Check minimum tokens
        if akk_count < min_akk_tokens or eng_count < min_eng_tokens:
            stats['filtered_too_short'] += 1
            continue
        
        # Check ratio
        ratio = akk_count / eng_count if eng_count > 0 else 0
        if ratio < min_ratio:
            stats['filtered_ratio_low'] += 1
            continue
        if ratio > max_ratio:
            stats['filtered_ratio_high'] += 1
            continue
        
        # Check for editorial content (reconstruct English text)
        if filter_editorial:
            eng_text = ' '.join([t for t in ex['eng_tokens'] if t != '<null>'])
            if is_editorial_content(eng_text):
                stats['filtered_editorial'] += 1
                continue
        
        cleaned.append(ex)
        stats['kept'] += 1
    
    if verbose:
        print(f"    Original: {stats['total']} examples")
        print(f"    Filtered (ratio < {min_ratio}): {stats['filtered_ratio_low']}")
        print(f"    Filtered (ratio > {max_ratio}): {stats['filtered_ratio_high']}")
        print(f"    Filtered (too short): {stats['filtered_too_short']}")
        print(f"    Filtered (editorial): {stats['filtered_editorial']}")
        print(f"    Kept: {stats['kept']} ({100*stats['kept']/max(stats['total'],1):.1f}%)")
    
    return cleaned


def load_train_data(form_to_lexeme, train_doc_ids, documents, clean_data=True):
    """Load training examples from structured dataset and manual alignments."""
    print("Loading training data for lexical model...")
    examples = []
    train_ids_set = set(train_doc_ids)
    
    # 1. Load Manually Aligned Segments (Gold Standard) - High weight
    print(f"  Loading manual alignments from {ALIGNED_SEGMENTS_FILE}...")
    manual_count = 0
    if os.path.exists(ALIGNED_SEGMENTS_FILE):
        with open(ALIGNED_SEGMENTS_FILE, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip(): continue
                try:
                    row = json.loads(line)
                    akk_text = replace_gaps(row['akk'])
                    eng_text = replace_gaps(row['eng'])
                    
                    raw_segment = tokenize_akkadian(akk_text)
                    akk_toks = [get_akkadian_representation(t, form_to_lexeme) for t in raw_segment]
                    eng_toks = tokenize_english(eng_text)
                    
                    if akk_toks and eng_toks:
                        examples.append({
                            "doc_id": row['doc_id'],
                            "akk_tokens": akk_toks + ['<null>'],
                            "eng_tokens": eng_toks + ['<null>'],
                            "weight": 10.0,  # High weight for manual data
                            "source": "manual"
                        })
                        manual_count += 1
                except: pass
    print(f"    Loaded {manual_count} manual alignments.")

    # 2. Load structured dataset
    for doc in documents:
        if doc['id'] not in train_ids_set:
            continue
            
        for part in doc['parts']:
            akk_text = replace_gaps(part['akk_text'])
            eng_text = replace_gaps(part['eng_text'])
            
            raw_segment = tokenize_akkadian(akk_text)
            akk_toks = [get_akkadian_representation(t, form_to_lexeme) for t in raw_segment]
            eng_toks = tokenize_english(eng_text)
            
            if akk_toks and eng_toks:
                examples.append({
                    "doc_id": doc['id'],
                    "akk_tokens": akk_toks + ['<null>'],
                    "eng_tokens": eng_toks + ['<null>'],
                    "weight": 1.0,
                    "source": "structured"
                })
    
    used_doc_ids = set(doc['id'] for doc in documents)
    print(f"  Loaded {len(examples)} sentence pairs from {len(used_doc_ids)} documents.")

    # 3. Load Full Documents (Weak Supervision) - docs without sentence splits
    print("  Loading additional full documents...")
    extra_docs_count = 0
    
    if os.path.exists(TRAIN_FILE):
        with open(TRAIN_FILE, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                doc_id = row['oare_id']
                if doc_id in used_doc_ids:
                    continue
                    
                full_akk = replace_gaps(row['transliteration'])
                full_eng = replace_gaps(row.get('translation', ''))
                
                if not full_akk or not full_eng:
                    continue
                    
                raw_segment = tokenize_akkadian(full_akk)
                akk_toks = [get_akkadian_representation(t, form_to_lexeme) for t in raw_segment]
                eng_toks = tokenize_english(full_eng)
                
                if akk_toks and eng_toks:
                    examples.append({
                        "doc_id": doc_id,
                        "akk_tokens": akk_toks + ['<null>'],
                        "eng_tokens": eng_toks + ['<null>'],
                        "weight": 1.0,
                        "source": "full_doc"
                    })
                    extra_docs_count += 1

    print(f"    Added {extra_docs_count} full documents.")
    print(f"  Total raw training examples: {len(examples)}")
    
    # 4. Clean training data (filter problematic pairs)
    if clean_data:
        examples = clean_training_examples(
            examples,
            min_ratio=0.2,   # Akkadian should be at least 20% of English tokens
            max_ratio=5.0,   # Akkadian should be at most 5x English tokens
            min_akk_tokens=2,
            min_eng_tokens=2,
            filter_editorial=True
        )
    
    print(f"  Final training examples: {len(examples)}")
    return examples

print("Training data loader with cleaning defined.")

In [ ]:
def compute_document_frequencies(examples, reverse=False):
    """Compute Document Frequency (DF) for tokens and pairs."""
    s_doc_counts = collections.defaultdict(set)
    t_doc_counts = collections.defaultdict(set)
    pair_doc_counts = collections.defaultdict(set)
    all_docs = set()
    
    for ex in examples:
        doc_id = ex.get('doc_id', 'unknown')
        all_docs.add(doc_id)
        
        source_tokens = ex['akk_tokens'] if reverse else ex['eng_tokens']
        target_tokens = ex['eng_tokens'] if reverse else ex['akk_tokens']
        
        s_set = set(source_tokens)
        t_set = set(target_tokens)
        
        for s in s_set:
            s_doc_counts[s].add(doc_id)
        for t in t_set:
            t_doc_counts[t].add(doc_id)
        for s in s_set:
            for t in t_set:
                pair_doc_counts[(s, t)].add(doc_id)
            
    s_df = {k: len(v) for k, v in s_doc_counts.items()}
    t_df = {k: len(v) for k, v in t_doc_counts.items()}
    pair_df = {k: len(v) for k, v in pair_doc_counts.items()}
    
    return s_df, t_df, pair_df, len(all_docs)

def initialize_model(examples, priors, s_df, t_df, pair_df, N, reverse=False, lexicon_types=None):
    """Initialize translation probabilities using PMI."""
    t = collections.defaultdict(lambda: collections.defaultdict(float))
    
    print("  Initializing probabilities with PMI...")
    co_occurrences = collections.defaultdict(set)
    for ex in examples:
        source_tokens = ex['akk_tokens'] if reverse else ex['eng_tokens']
        target_tokens = ex['eng_tokens'] if reverse else ex['akk_tokens']
        
        for s in source_tokens:
            for tgt in target_tokens:
                co_occurrences[s].add(tgt)
                
    for s in co_occurrences:
        possible_targets = co_occurrences[s]
        total_weight = 0.0
        weights = {}
        s_freq = s_df.get(s, 1)
        
        for tgt in possible_targets:
            if tgt == '<null>':
                weight = 10.0
            else:
                t_freq = t_df.get(tgt, 1)
                pair_freq = pair_df.get((s, tgt), 0)
                pmi_score = (pair_freq * N) / (s_freq * t_freq + 1e-6)
                weight = pmi_score ** 3
            
            # Apply Type Constraints
            if lexicon_types:
                if reverse:
                    s_type = lexicon_types.get(s)
                    penalty = get_type_penalty(tgt, s, None, s_type, reverse=True)
                else:
                    t_type = lexicon_types.get(tgt)
                    penalty = get_type_penalty(s, tgt, None, t_type, reverse=False)
                weight *= penalty
            
            # Phonetic Boost for proper nouns
            if not reverse and s[0].isupper() and s.isalpha() and len(s) > 3:
                norm_s = normalize_akkadian_string(s).lower().replace('-', '')
                norm_tgt = normalize_akkadian_string(tgt).lower().replace('-', '')
                
                if len(norm_tgt) > 3:
                    if soundex(norm_s) == soundex(norm_tgt):
                        weight *= 100.0
                    elif difflib.SequenceMatcher(None, norm_s, norm_tgt).ratio() > 0.8:
                        weight *= 50.0

            # Dictionary priors
            if reverse:
                if s in priors and tgt in priors[s]:
                    weight *= 100.0
            else:
                if tgt in priors and s in priors[tgt]:
                    weight *= 100.0
                    
            weights[tgt] = weight
            total_weight += weight
            
        for tgt in possible_targets:
            if total_weight > 0:
                t[s][tgt] = weights[tgt] / total_weight
            else:
                t[s][tgt] = 1.0 / len(possible_targets)
    return t

print("Model initialization functions defined.")

In [ ]:
def train_joint_lexical_models(examples, priors, iterations=ITERATIONS, lexicon_types=None):
    """Train Joint Lexical Models (IBM1) with EM and bidirectional agreement."""
    print(f"Training Joint Lexical Models (IBM1) for {iterations} iterations...")
    
    # Initialize both models using PMI
    # Forward: eng -> akk
    s_df, t_df, pair_df, N = compute_document_frequencies(examples, reverse=False)
    print("Initializing Forward model (eng->akk)...")
    t_fwd = initialize_model(examples, priors, s_df, t_df, pair_df, N, reverse=False, lexicon_types=lexicon_types)
    
    # Reverse: akk -> eng
    s_df_rev, t_df_rev, pair_df_rev, N_rev = compute_document_frequencies(examples, reverse=True)
    print("Initializing Reverse model (akk->eng)...")
    t_rev = initialize_model(examples, priors, s_df_rev, t_df_rev, pair_df_rev, N_rev, reverse=True, lexicon_types=lexicon_types)
    
    # EM Loop with Agreement
    for it in range(iterations):
        print(f"  Iteration {it+1}/{iterations}...")
        count_fwd_pair = collections.defaultdict(float)
        count_fwd_source = collections.defaultdict(float)
        count_rev_pair = collections.defaultdict(float)
        count_rev_source = collections.defaultdict(float)
        
        for ex in examples:
            eng_tokens = ex['eng_tokens']
            akk_tokens = ex['akk_tokens']
            weight = ex['weight']
            
            # Forward Updates (Eng -> Akk)
            for e in eng_tokens:
                Z = sum(t_fwd[e][a] * t_rev[a][e] for a in akk_tokens)
                if Z > 0:
                    for a in akk_tokens:
                        score = t_fwd[e][a] * t_rev[a][e]
                        delta = (weight * score) / Z
                        count_fwd_pair[(e, a)] += delta
                        count_fwd_source[e] += delta

            # Reverse Updates (Akk -> Eng)
            for a in akk_tokens:
                Z = sum(t_rev[a][e] * t_fwd[e][a] for e in eng_tokens)
                if Z > 0:
                    for e in eng_tokens:
                        score = t_rev[a][e] * t_fwd[e][a]
                        delta = (weight * score) / Z
                        count_rev_pair[(a, e)] += delta
                        count_rev_source[a] += delta
                    
        # M-Step: Update probabilities
        for e, a_map in t_fwd.items():
            total = count_fwd_source[e]
            if total == 0: continue
            for a in a_map:
                t_fwd[e][a] = count_fwd_pair.get((e, a), 0.0) / total
        
        for a, e_map in t_rev.items():
            total = count_rev_source[a]
            if total == 0: continue
            for e in e_map:
                t_rev[a][e] = count_rev_pair.get((a, e), 0.0) / total

    print("Training complete.")
    return t_fwd, t_rev

def prune_and_save_lexical_model(forward_model, reverse_model, output_file):
    """Prune low-probability translations and save models."""
    print("Pruning and saving lexical models...")
    
    GARBAGE = {'-', '.', '(', ')', '[', ']', '...', '..', ','}

    def prune(model):
        final = {}
        for s, t_map in model.items():
            if s in GARBAGE: continue
            sorted_targets = sorted(t_map.items(), key=lambda x: x[1], reverse=True)
            kept = {}
            for i, (tgt, p) in enumerate(sorted_targets):
                if i >= TOP_K: break
                if p < MIN_PROB: break
                if tgt in GARBAGE: continue
                kept[tgt] = p
            if kept:
                final[s] = kept
        return final

    final_output = {
        "forward": prune(forward_model),
        "reverse": prune(reverse_model)
    }
            
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(final_output, f, ensure_ascii=False, indent=2)
    print(f"  Saved lexical models to {output_file}")
    return final_output

print("Lexical model training functions defined.")

### 1.7 Length Model Estimation

Estimate log-normal statistics for Akkadian/English sentence length ratios from training data.

In [ ]:
def compute_length_stats(train_doc_ids, documents):
    """Compute log-normal length ratio statistics from training set."""
    print("Computing length statistics from TRAIN set...")
    train_ids_set = set(train_doc_ids)
    log_ratios = []
    
    for doc in documents:
        if doc['id'] not in train_ids_set:
            continue
            
        for part in doc['parts']:
            akk_words = tokenize_akkadian_for_indexing(part['akk_text'])
            n_akk_words = len(akk_words)
            
            eng_toks = tokenize_english(part['eng_text'])
            n_eng_toks = len(eng_toks)
            
            if n_eng_toks > 0 and n_akk_words > 0:
                log_ratios.append(math.log(n_akk_words / n_eng_toks))

    if log_ratios:
        mu_log = sum(log_ratios) / len(log_ratios)
        variance = sum((x - mu_log) ** 2 for x in log_ratios) / len(log_ratios)
        sigma_log = math.sqrt(variance)
    else:
        mu_log = 0.0
        sigma_log = 1.0
        
    print(f"  Stats from {len(log_ratios)} sentence pairs:")
    print(f"    Mu_log={mu_log:.3f}, Sigma_log={sigma_log:.3f}")
    print(f"    Geometric Mean Ratio: {math.exp(mu_log):.2f}")
    
    stats = {"mu_log": mu_log, "sigma_log": sigma_log}
    
    with open(LENGTH_MODEL_FILE, 'w', encoding='utf-8') as f:
        json.dump(stats, f, indent=2)
    print(f"  Saved length model to {LENGTH_MODEL_FILE}")
    
    return stats

print("Length model function defined.")

### 1.8 Sentence Boundary Inference

Predict Akkadian sentence boundaries given English translations using lexical scores and length priors.

In [ ]:
def get_lexical_score(eng_tokens, akk_lexemes, forward_model, reverse_model, mode='symmetric'):
    """
    Compute lexical alignment score between English and Akkadian tokens.
    mode='recall': Score how well English is covered by Akkadian.
    mode='symmetric': Score both directions.
    """
    score = 0.0
    epsilon = 1e-6
    
    # Recall: For each English word, find best Akkadian match
    recall_score = 0.0
    for e in eng_tokens:
        best_pair_score = epsilon
        for a in akk_lexemes:
            p_fwd = forward_model.get(e, {}).get(a, epsilon)
            p_rev = reverse_model.get(a, {}).get(e, epsilon)
            pair_score = math.sqrt(p_fwd * p_rev)
            if pair_score > best_pair_score:
                best_pair_score = pair_score
        recall_score += math.log(best_pair_score)
        
    if mode == 'recall':
        return recall_score

    # Precision: For each Akkadian word, find best English match
    precision_score = 0.0
    if akk_lexemes:
        for a in akk_lexemes:
            best_pair_score = epsilon
            for e in eng_tokens:
                p_fwd = forward_model.get(e, {}).get(a, epsilon)
                p_rev = reverse_model.get(a, {}).get(e, epsilon)
                pair_score = math.sqrt(p_fwd * p_rev)
                if pair_score > best_pair_score:
                    best_pair_score = pair_score
            precision_score += math.log(best_pair_score)
            
    return recall_score + precision_score

def predict_sentence_boundary(eng_tokens, akk_words, start_index, lexical_model, length_model, form_to_lexeme, next_eng_tokens=None):
    """
    Predict the end index (whitespace-based) for the Akkadian span.
    Uses Contrastive Scoring if next_eng_tokens is provided.
    """
    if not eng_tokens:
        return start_index
        
    n_eng = len(eng_tokens)
    
    # Use Log-Normal model
    mu_log = length_model.get('mu_log', 0.0)
    sigma_log = length_model.get('sigma_log', 1.0)
    
    # Define search window
    ratio_upper = math.exp(mu_log + 3 * sigma_log)
    ratio_lower = math.exp(mu_log - 3 * sigma_log)
    
    max_len = int(n_eng * ratio_upper) + 10
    min_len = max(0, int(n_eng * ratio_lower) - 5)
    
    max_possible_end = min(len(akk_words), start_index + max_len)
    min_possible_end = min(len(akk_words), start_index + min_len)
    
    best_score = -float('inf')
    best_end = start_index + 1
    
    forward_model = lexical_model.get('forward', {})
    reverse_model = lexical_model.get('reverse', {})
    
    for end_idx in range(min_possible_end, max_possible_end + 1):
        span_words = akk_words[start_index:end_idx]
        span_text = " ".join(span_words)
        
        span_tokens = tokenize_akkadian(span_text)
        span_lexemes = [get_akkadian_representation(t, form_to_lexeme) for t in span_tokens]
        
        # 1. Match Score (Current English <-> Current Akkadian)
        match_score = get_lexical_score(eng_tokens, span_lexemes, forward_model, reverse_model, mode='recall')
        
        # 2. Contrastive Score (penalize if span matches NEXT sentence)
        contrast_score = 0.0
        if next_eng_tokens:
            contrast_score = get_lexical_score(next_eng_tokens, span_lexemes, forward_model, reverse_model, mode='recall')
            
        # 3. Length Prior (Log-Normal)
        current_len = end_idx - start_index
        ratio = current_len / n_eng if n_eng > 0 else 0
        
        if ratio > 0:
            ln_x = math.log(ratio)
            length_score = -ln_x - ((ln_x - mu_log) ** 2) / (2 * sigma_log ** 2)
        else:
            length_score = -100.0
        
        # Total Score
        total_score = match_score + (length_score * 2.0)
        
        if next_eng_tokens:
            total_score -= (contrast_score * 0.5)
        
        if total_score > best_score:
            best_score = total_score
            best_end = end_idx
            
    return best_end

print("Inference functions defined.")

### 1.9 Evaluation

Evaluate sentence boundary prediction accuracy on held-out dev/test sets.

In [ ]:
def evaluate_split(split_name, doc_ids, documents, lexical_model, length_model, form_to_lexeme):
    """Evaluate sentence boundary prediction on a data split."""
    print(f"Evaluating on {split_name} set ({len(doc_ids)} docs)...")
    doc_ids_set = set(doc_ids)
    
    errors = []
    within_1 = 0
    within_3 = 0
    total_sents = 0
    
    for doc in documents:
        if doc['id'] not in doc_ids_set:
            continue
        
        oare_words = tokenize_akkadian_for_indexing(doc['akk_text'])
        current_word_idx = 0
        
        for i, part in enumerate(doc['parts']):
            part_akk_words = tokenize_akkadian_for_indexing(part['akk_text'])
            gold_len = len(part_akk_words)
            gold_end_idx = current_word_idx + gold_len
            
            eng_tokens = tokenize_english(part['eng_text'])
            
            # Get next sentence tokens for contrastive scoring
            next_eng_tokens = None
            if i + 1 < len(doc['parts']):
                next_eng_tokens = tokenize_english(doc['parts'][i+1]['eng_text'])
            
            # Predict
            predicted_end = predict_sentence_boundary(
                eng_tokens, oare_words, current_word_idx, 
                lexical_model, length_model, form_to_lexeme,
                next_eng_tokens=next_eng_tokens
            )
            
            error = abs(predicted_end - gold_end_idx)
            errors.append(error)
            
            if error <= 1: within_1 += 1
            if error <= 3: within_3 += 1
            total_sents += 1
            
            current_word_idx += gold_len
                
    if not errors:
        print("  No sentences evaluated.")
        return {}

    results = {
        'split': split_name,
        'total_sentences': total_sents,
        'mean_error': statistics.mean(errors),
        'median_error': statistics.median(errors),
        'accuracy_within_1': (within_1 / total_sents) * 100,
        'accuracy_within_3': (within_3 / total_sents) * 100
    }
    
    print(f"  Results for {split_name}:")
    print(f"    Total Sentences: {results['total_sentences']}")
    print(f"    Mean Absolute Boundary Error: {results['mean_error']:.2f} words")
    print(f"    Median Error: {results['median_error']:.2f} words")
    print(f"    Accuracy (within ±1 word): {results['accuracy_within_1']:.2f}%")
    print(f"    Accuracy (within ±3 words): {results['accuracy_within_3']:.2f}%")
    
    return results

print("Evaluation function defined.")

In [ ]:
def run_full_pipeline():
    """Execute the complete Akkadian-English alignment pipeline."""
    print("=" * 60)
    print("AKKADIAN-ENGLISH SENTENCE ALIGNER PIPELINE")
    print("=" * 60)
    
    # Step 1: Load Dataset
    print("\n" + "=" * 40)
    print("STEP 1: Loading Dataset")
    print("=" * 40)
    documents = load_dataset()
    
    if not documents:
        print("ERROR: No documents loaded. Check data files.")
        return None
    
    # Step 2: Create Splits
    print("\n" + "=" * 40)
    print("STEP 2: Creating Data Splits")
    print("=" * 40)
    splits = create_splits(documents)
    
    # Step 3: Load Lexicon Resources
    print("\n" + "=" * 40)
    print("STEP 3: Loading Lexicon Resources")
    print("=" * 40)
    form_to_lexeme = load_lexicon()
    lexicon_types = load_lexicon_types()
    priors = load_dictionary_priors(form_to_lexeme)
    
    # Step 4: Train Lexical Model
    print("\n" + "=" * 40)
    print("STEP 4: Training Lexical Model (IBM Model 1)")
    print("=" * 40)
    examples = load_train_data(form_to_lexeme, splits['train'], documents)
    
    if not examples:
        print("ERROR: No training examples found.")
        return None
        
    forward_model, reverse_model = train_joint_lexical_models(examples, priors, lexicon_types=lexicon_types)
    lexical_model = prune_and_save_lexical_model(forward_model, reverse_model, LEXICAL_MODEL_FILE)
    
    # Step 5: Estimate Length Model
    print("\n" + "=" * 40)
    print("STEP 5: Estimating Length Model")
    print("=" * 40)
    length_model = compute_length_stats(splits['train'], documents)
    
    # Step 6: Evaluate
    print("\n" + "=" * 40)
    print("STEP 6: Evaluation")
    print("=" * 40)
    dev_results = evaluate_split("DEV", splits['dev'], documents, lexical_model, length_model, form_to_lexeme)
    test_results = evaluate_split("TEST", splits['test'], documents, lexical_model, length_model, form_to_lexeme)
    
    print("\n" + "=" * 60)
    print("PIPELINE COMPLETE")
    print("=" * 60)
    
    return {
        'documents': documents,
        'splits': splits,
        'lexical_model': lexical_model,
        'length_model': length_model,
        'form_to_lexeme': form_to_lexeme,
        'dev_results': dev_results,
        'test_results': test_results
    }

print("Full pipeline function defined.")

In [ ]:
# Run the full pipeline
results = run_full_pipeline()

### 1.10 Load Pre-trained Models (Optional)

Skip training and load previously saved models for inference.

In [ ]:
def load_pretrained_models():
    """Load pre-trained models and data without retraining."""
    print("Loading pre-trained models...")
    
    # Load dataset
    documents = load_dataset()
    
    # Load splits
    if os.path.exists(SPLITS_FILE):
        with open(SPLITS_FILE, 'r') as f:
            splits = json.load(f)
        print(f"  Loaded splits: Train={len(splits['train'])}, Dev={len(splits['dev'])}, Test={len(splits['test'])}")
    else:
        print(f"  Warning: {SPLITS_FILE} not found. Creating new splits...")
        splits = create_splits(documents)
    
    # Load lexicon
    form_to_lexeme = load_lexicon()
    
    # Load lexical model
    if os.path.exists(LEXICAL_MODEL_FILE):
        with open(LEXICAL_MODEL_FILE, 'r') as f:
            lexical_model = json.load(f)
        print(f"  Loaded lexical model from {LEXICAL_MODEL_FILE}")
    else:
        print(f"  Warning: {LEXICAL_MODEL_FILE} not found.")
        lexical_model = {"forward": {}, "reverse": {}}
    
    # Load length model
    if os.path.exists(LENGTH_MODEL_FILE):
        with open(LENGTH_MODEL_FILE, 'r') as f:
            length_model = json.load(f)
        print(f"  Loaded length model from {LENGTH_MODEL_FILE}")
    else:
        print(f"  Warning: {LENGTH_MODEL_FILE} not found.")
        length_model = {"mu_log": 0.0, "sigma_log": 1.0}
    
    return {
        'documents': documents,
        'splits': splits,
        'lexical_model': lexical_model,
        'length_model': length_model,
        'form_to_lexeme': form_to_lexeme
    }

# Uncomment to load pre-trained models instead of running full pipeline
# results = load_pretrained_models()

### 1.11 Inspect Lexical Model

Explore learned word translations and model statistics.

In [ ]:
def lookup_translation(word, results, direction='eng_to_akk', top_n=10):
    """
    Look up translations for a word in the lexical model.
    direction: 'eng_to_akk' or 'akk_to_eng'
    """
    if results is None:
        print("Run the pipeline first!")
        return
    
    lexical_model = results['lexical_model']
    
    if direction == 'eng_to_akk':
        model = lexical_model.get('forward', {})
        word = word.lower()
    else:
        model = lexical_model.get('reverse', {})
    
    if word not in model:
        print(f"Word '{word}' not found in model.")
        return
    
    translations = sorted(model[word].items(), key=lambda x: x[1], reverse=True)[:top_n]
    
    print(f"Translations for '{word}' ({direction}):")
    for target, prob in translations:
        print(f"  {target}: {prob:.4f}")

def show_model_stats(results):
    """Display statistics about the lexical model."""
    if results is None:
        print("Run the pipeline first!")
        return
    
    lexical_model = results['lexical_model']
    
    fwd = lexical_model.get('forward', {})
    rev = lexical_model.get('reverse', {})
    
    print("Lexical Model Statistics:")
    print(f"  Forward (Eng->Akk): {len(fwd)} English words")
    print(f"  Reverse (Akk->Eng): {len(rev)} Akkadian words")
    
    # Count total translation pairs
    fwd_pairs = sum(len(v) for v in fwd.values())
    rev_pairs = sum(len(v) for v in rev.values())
    
    print(f"  Forward translation pairs: {fwd_pairs}")
    print(f"  Reverse translation pairs: {rev_pairs}")

print("Inspection utilities defined.")

In [ ]:
# Show model statistics
show_model_stats(results)

# Example: Look up translations
# lookup_translation('silver', results, direction='eng_to_akk')
# lookup_translation('kaspum', results, direction='akk_to_eng')

---

## Part 2: Global Alignment Pipeline

This section implements the production alignment system using Viterbi dynamic programming for globally optimal sentence alignment.

**Key Components:**
- **AlignmentScorer**: Multi-feature scoring (translation probabilities, IDF, phonetics, length priors)
- **Anchor Detection**: High-confidence alignment points (proper nouns, gaps, numbers)
- **Viterbi DP**: Dynamic programming for globally optimal alignment with diagonal prior
- **Quality Filtering**: Score thresholds, length bounds, partial alignment extraction

### 2.1 Advanced Text Cleaning

Additional text processing: Akkadian heuristic sets (pronouns, particles) and English sentence splitting.

In [ ]:
# Heuristic Sets for Alignment Constraints
INDEPENDENT_PRONOUNS = {
    'anāku', 'a-na-ku', 'a-na-ku₄', 
    'attā', 'a-ta', 'a-tá',
    'atti', 'a-tí',
    'šū', 'šu-ú',
    'šī', 'ší-i',
    'nīnu', 'ni-nu', 'né-nu',
    'attunu', 'a-tù-nu',
    'šunu', 'šu-nu',
    'šina', 'ší-na'
}

CLAUSE_PARTICLES = {
    'lā', 'la', 'lá', 'ul', 'ú-ul',
    'umma', 'um-ma',
    'ma', '-ma'
}

# Stopwords for constraint filtering
STOPWORDS = {
    'of', 'and', 'to', 'in', 'a', 'is', 'was', 'for', 'on', 'with', 'by', 'from', 
    'it', 'he', 'she', 'they', 'we', 'you', 'i', 'me', 'us', 'him', 'them',
    'my', 'his', 'her', 'their', 'our', 'your', 'its',
    'myself', 'yourself', 'himself', 'herself', 'themselves', 'ourselves', 'yourselves',
    'say', 'speak', 'be', 'are', 'will', 'shall', 'have', 'has', 'had', 'do', 'does', 'did', 
    'an', 'at', 'as', 'but', 'or', 'not', 'if', 'when', 'thus', 'so', 'the'
}

def clean_akkadian(text):
    """
    Clean Akkadian text with exploded tokenization.
    Reverses 'ma - nu' -> 'ma-nu', '( d )' -> '(d)', '0 . 5' -> '0.5'.
    """
    if not text:
        return text
    # Collapse spaced hyphens/dots between word characters
    text = re.sub(r'(?<=[\w\d])\s+-\s+(?=[\w\d])', '-', text)
    text = re.sub(r'(?<=[\w\d])\s+\.\s+(?=[\w\d])', '.', text)
    
    # Collapse spaced parentheses
    text = re.sub(r'\s*\(\s*', '(', text)
    text = re.sub(r'\s*\)\s*', ')', text)
    
    # Apply gap replacement
    text = replace_gaps(text)
    return text

def strip_akkadian_suffixes(norm_word, is_valid_word_fn=None):
    """
    Strip common Akkadian suffixes from a NORMALIZED string.
    Handles case endings, mimation, enclitics, and possessive suffixes.
    """
    if not norm_word or len(norm_word) < 4:
        return norm_word
    
    w = norm_word
    
    # 1. Enclitics (ma, ni, etc.)
    enclitics = ['ma']
    for suffix in enclitics:
        if w.endswith(suffix):
            stem = w[:-len(suffix)]
            if len(stem) >= 2:
                if is_valid_word_fn:
                    if is_valid_word_fn(stem):
                        w = stem
                        break
                elif len(stem) >= 3:
                    w = stem
                    break

    # 2. Pronominal Suffixes (Possessive)
    possessives = ['sunu', 'sina', 'kunu', 'kina', 'su', 'sa', 'ka', 'ki', 'ya', 'ni']
    for suffix in possessives:
        if w.endswith(suffix) and len(w) - len(suffix) >= 2:
            w = w[:-len(suffix)]
            break
    
    # 3. Mimation (um, am, im)
    if w.endswith('um') or w.endswith('am') or w.endswith('im'):
        if len(w) >= 4: 
            w = w[:-2]
            
    # 4. Case Vowels (u, a, i)
    if w.endswith('u') or w.endswith('a') or w.endswith('i'):
        if len(w) >= 3:
             w = w[:-1]
             
    # 5. Dual/Plural markers (an, in)
    if w.endswith('an') or w.endswith('in'):
        if len(w) >= 4:
            w = w[:-2]
            
    return w

def edit_distance(s1, s2):
    """Compute Levenshtein edit distance."""
    if len(s1) < len(s2):
        return edit_distance(s2, s1)
    if len(s2) == 0:
        return len(s1)
    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row
    return previous_row[-1]

def split_english_sentences(text):
    """Split English text into sentences."""
    if not text:
        return []
    
    # Protect abbreviations
    text = text.replace("Mr.", "Mr<DOT>")
    text = text.replace("Dr.", "Dr<DOT>")
    text = text.replace("St.", "St<DOT>")
    text = text.replace("No.", "No<DOT>")
    text = text.replace("vs.", "vs<DOT>")
    
    # Split on sentence-ending punctuation followed by capital letter
    parts = re.split(r'(?<=[.?!])\s+(?=[A-Z0-9"\(])', text)
    
    sentences = []
    for p in parts:
        p = p.strip()
        if p:
            p = p.replace("<DOT>", ".")
            sentences.append(p)
    return sentences

print("Advanced text cleaning utilities loaded.")

### 2.2 AlignmentScorer Class

Core scoring class computing alignment quality using:
- Bidirectional translation probabilities (IBM Model 1)
- IDF weighting for discriminative terms
- Fuzzy/phonetic matching (Soundex for proper nouns)
- Log-normal length ratio prior
- Boundary heuristics (clause starters)

In [ ]:
class AlignmentScorer:
    """
    Scoring class for computing alignment scores between English and Akkadian spans.
    Uses bidirectional translation model, IDF weighting, fuzzy matching, and length priors.
    """
    def __init__(self, lexical_model, form_to_lexeme, idf_counts=None, total_docs=0, mu_log=0.0, sigma_log=1.0):
        self.lexicon = form_to_lexeme
        self.definitions = {}  # Can be loaded separately if available
        
        # Load Forward and Reverse models
        if isinstance(lexical_model, dict):
            self.forward_probs = lexical_model.get('forward', {})
            self.reverse_probs = lexical_model.get('reverse', {})
        else:
            self.forward_probs = {}
            self.reverse_probs = {}
            
        self.epsilon = 1e-6
        self.min_prob = 0.001
        
        # IDF Calculation
        self.idf = {}
        if idf_counts and total_docs > 0:
            for word, count in idf_counts.items():
                self.idf[word] = math.log(total_docs / (count + 1))
        else:
            self.idf = defaultdict(lambda: 1.0)

        # Length stats (Log-Normal)
        self.mu_log = mu_log
        self.sigma_log = sigma_log
        
        # Caches
        self.fuzzy_cache = {}
        self.span_cache = {}
        
        self.akk_vocab = list(self.reverse_probs.keys()) if self.reverse_probs else []
        
        # Bucket vocab by first letter for fast fuzzy matching
        self.akk_vocab_by_letter = defaultdict(list)
        for w in self.akk_vocab:
            if w:
                self.akk_vocab_by_letter[w[0]].append(w)

    def is_valid_word(self, word):
        """Check if a normalized word exists in the lexicon or reverse model."""
        if not word:
            return False
        return word in self.lexicon or word in self.reverse_probs

    def clear_cache(self):
        """Clear the span score cache."""
        self.span_cache = {}

    def get_fuzzy_matches(self, token):
        """Get fuzzy matches for unknown tokens."""
        if token in self.fuzzy_cache:
            return self.fuzzy_cache[token]
            
        if len(token) < 4:
            return []
            
        first_char = token[0]
        candidates = self.akk_vocab_by_letter.get(first_char, [])
        
        if not candidates:
            return []
            
        matches = difflib.get_close_matches(token, candidates, n=5, cutoff=0.8)
        self.fuzzy_cache[token] = matches
        return matches

    def precompute_akkadian_props(self, akk_tokens):
        """Precompute properties for Akkadian tokens for fast scoring."""
        props = []
        for t in akk_tokens:
            candidates = set()
            norm = normalize_akkadian_string(t)
            candidates.add(norm)
            
            # Add lexicon lemma
            lex = self.lexicon.get(norm)
            if lex:
                candidates.add(lex)
            candidates.add(t)
            
            # IMPROVEMENT: Add dehyphenated form if it's in the lexical model
            # This helps with cases like 'i-na' -> 'ina' (preposition)
            # which the lexicon might incorrectly map to 'Innā' (personal name)
            dehyphenated = t.replace('-', '')
            if dehyphenated != t and dehyphenated in self.reverse_probs:
                candidates.add(dehyphenated)
            
            # Also try dehyphenated normalized form
            norm_dehyph = norm.replace('-', '')
            if norm_dehyph != norm and norm_dehyph in self.reverse_probs:
                candidates.add(norm_dehyph)
            
            is_unknown = all(c not in self.reverse_probs for c in candidates)
            
            fuzzy_candidates = []
            if is_unknown:
                fuzzy_matches = self.get_fuzzy_matches(norm)
                if fuzzy_matches:
                    fuzzy_candidates = fuzzy_matches
            
            cand_soundex = {c: soundex(c) for c in candidates}
            stem = strip_akkadian_suffixes(norm, self.is_valid_word)
            
            props.append({
                'candidates': list(candidates),
                'fuzzy_candidates': fuzzy_candidates,
                'cand_soundex': cand_soundex,
                'stem': stem
            })
        return props

    def score_span(self, eng_tokens, akk_tokens, akk_props=None, include_ratio_penalty=True, 
                   eng_text=None, proper_noun_bonus=3.0, sumerogram_bonus=2.0):
        """
        Score a candidate Akkadian span against English tokens.
        Returns: alignment score (higher is better)
        
        Uses internal caching for repeated calls with same tokens.
        
        Args:
            eng_tokens: List of English tokens (lowercased)
            akk_tokens: List of Akkadian tokens
            akk_props: Precomputed properties for Akkadian tokens
            include_ratio_penalty: Whether to include length ratio penalty
            eng_text: Original English text (for proper noun detection)
            proper_noun_bonus: Bonus per proper noun match (+3.0 default)
            sumerogram_bonus: Bonus per Sumerogram match (+2.0 default)
        """
        if not eng_tokens:
            return -100.0
        
        # Create cache key from tokens (immutable tuple)
        cache_key = (tuple(eng_tokens), tuple(akk_tokens), include_ratio_penalty)
        if cache_key in self.span_cache:
            return self.span_cache[cache_key]
        
        # Precompute props if not provided
        if akk_props is None:
            akk_props = self.precompute_akkadian_props(akk_tokens)
        
        # 1. Coverage Score (Greedy 1-to-1 Matching)
        coverage_log_prob = 0.0
        used_akk_indices = set()
        
        # Sort English by IDF (rare words first)
        eng_indices = sorted(range(len(eng_tokens)), 
                            key=lambda i: self.idf.get(eng_tokens[i], 0.0), reverse=True)
        
        for e_idx in eng_indices:
            e = eng_tokens[e_idx]
            e_lower = e.lower()
            
            if e_lower == 'the':
                continue
                
            max_score = self.epsilon
            best_a_idx = -1
            is_proper = e[0].isupper() and e[0].isalpha()

            # Gap handling
            if e in ['<big_gap>', '<gap>']:
                for a_idx, props in enumerate(akk_props):
                    if a_idx in used_akk_indices:
                        continue
                    if e in props['candidates']:
                        best_a_idx = a_idx
                        max_score = 100.0
                        break
                
                if best_a_idx != -1:
                    used_akk_indices.add(best_a_idx)
                    coverage_log_prob += math.log(max_score)
                    continue

            # Find best match in span
            for a_idx, props in enumerate(akk_props):
                if a_idx in used_akk_indices:
                    continue
                    
                candidates = props['candidates']
                fuzzy_candidates = props['fuzzy_candidates']
                cand_soundex = props['cand_soundex']
                akk_stem = props['stem']
                
                best_cand_score = 0.0
                
                # 1. Exact/Lexicon Matches
                for cand in candidates:
                    p_fwd = self.forward_probs.get(e_lower, {}).get(cand, 0.0)
                    p_rev = self.reverse_probs.get(cand, {}).get(e_lower, 0.0) if self.reverse_probs else 0.0
                    
                    # Prefer bidirectional agreement, fallback to single direction
                    if p_fwd > 0 and p_rev > 0:
                        score = math.sqrt(p_fwd * p_rev)  # Both directions agree
                    elif p_fwd > 0:
                        score = p_fwd * 0.7  # Only forward evidence (penalize slightly)
                    elif p_rev > 0:
                        score = p_rev * 0.7  # Only reverse evidence (penalize slightly)
                    else:
                        score = 0.0
                    
                    if score > best_cand_score:
                        best_cand_score = score
                        
                # 2. Stem Match (for non-proper nouns)
                if not is_proper and best_cand_score < 0.1:
                    translations = self.forward_probs.get(e_lower, {})
                    for trans, prob in translations.items():
                        if prob < 0.01:
                            continue
                        if not isinstance(trans, str):
                            trans = str(trans)
                        trans_stem = strip_akkadian_suffixes(
                            normalize_akkadian_string(trans), 
                            is_valid_word_fn=self.is_valid_word
                        )
                        if trans_stem == akk_stem:
                            stem_score = 0.8 * prob
                            if stem_score > best_cand_score:
                                best_cand_score = stem_score

                # 3. Fuzzy Matches
                if best_cand_score < self.epsilon and fuzzy_candidates:
                    fuzzy_score_sum = 0.0
                    for f_cand in fuzzy_candidates:
                        p_fwd = self.forward_probs.get(e_lower, {}).get(f_cand, 0.0)
                        p_rev = self.reverse_probs.get(f_cand, {}).get(e_lower, 0.0) if self.reverse_probs else 0.0
                        
                        # Prefer bidirectional, fallback to single direction
                        if p_fwd > 0 and p_rev > 0:
                            s = math.sqrt(p_fwd * p_rev)
                        elif p_fwd > 0:
                            s = p_fwd * 0.7
                        elif p_rev > 0:
                            s = p_rev * 0.7
                        else:
                            s = 0.0
                        fuzzy_score_sum += s
                    
                    avg_fuzzy_score = fuzzy_score_sum / len(fuzzy_candidates)
                    ambiguity_factor = 1.0 / (1.0 + math.log(len(fuzzy_candidates)))
                    final_fuzzy_score = avg_fuzzy_score * 0.5 * ambiguity_factor
                    
                    if final_fuzzy_score > best_cand_score:
                        best_cand_score = final_fuzzy_score
                
                # 4. Phonetic Fallback (Soundex) - for proper nouns
                if best_cand_score < 0.1 and is_proper:
                    e_soundex = soundex(e)
                    for cand in candidates:
                        c_soundex = cand_soundex.get(cand)
                        if c_soundex and c_soundex == e_soundex:
                            if 0.4 > best_cand_score:
                                best_cand_score = 0.4
                                break

                if best_cand_score > max_score:
                    max_score = best_cand_score
                    best_a_idx = a_idx
            
            # Mark as used if good match
            threshold = 0.02
            if e_lower in STOPWORDS:
                threshold = 0.5
            
            if best_a_idx != -1 and max_score > threshold:
                used_akk_indices.add(best_a_idx)
            
            # Use a reasonable floor for unknown tokens instead of epsilon
            # This prevents extreme negative scores when Akkadian tokens aren't in the model
            # A score of 0.05 gives log(0.05) ≈ -3 instead of log(1e-6) ≈ -13.8
            score_for_log = max(max_score, 0.05)
            
            weight = max(self.idf.get(e, 1.0), 1.0)
            coverage_log_prob += weight * math.log(score_for_log)
            
        # 2. Length Penalty (unused tokens)
        num_unused = len(akk_tokens) - len(used_akk_indices)
        length_penalty = -0.5 * num_unused
        
        # 3. Log-Normal Length Ratio Penalty
        ratio_penalty = 0.0
        if include_ratio_penalty:
            n_akk = len(akk_tokens)
            n_eng = len(eng_tokens)
            
            if n_akk > 0 and n_eng > 0:
                log_ratio = math.log(n_akk / n_eng)
                z = (log_ratio - self.mu_log) / self.sigma_log
                ratio_penalty = -1.0 * (z ** 2)
            elif n_akk == 0 and n_eng == 0:
                ratio_penalty = 0.0
            else:
                ratio_penalty = -10.0 - (n_eng * 3.0)

        # 4. Boundary Heuristics
        boundary_score = 0.0
        start_words = {'a-na', 'um-ma', 'kīma', 'ša', 'i-na', 'šumma', 'šu-ma'}
        
        if akk_tokens:
            if akk_tokens[-1] in start_words:
                boundary_score -= 5.0

        # 5. Proper Noun Bonus (using original text for case detection)
        proper_noun_score = 0.0
        matched_proper_nouns = set()
        
        if eng_text:
            proper_nouns = extract_proper_nouns_from_text(eng_text)
            
            for pn in proper_nouns:
                if pn in matched_proper_nouns:
                    continue
                    
                pn_norm = normalize_for_soundex(pn)
                pn_sdx = soundex(pn_norm)
                if not pn_sdx:
                    continue
                
                # Look for Soundex match in Akkadian span
                for a_tok in akk_tokens:
                    akk_norm = normalize_for_soundex(a_tok)
                    akk_sdx = soundex(akk_norm)
                    
                    if akk_sdx and pn_sdx == akk_sdx:
                        proper_noun_score += proper_noun_bonus
                        matched_proper_nouns.add(pn)
                        break

        # 6. Sumerogram Bonus
        sumerogram_score = 0.0
        for e_tok in eng_tokens:
            e_lower = e_tok.lower()
            if e_lower in STOPWORDS:
                continue
            
            for a_tok in akk_tokens:
                a_upper = a_tok.upper()
                if a_upper in SUMEROGRAM_TO_ENGLISH:
                    translations = SUMEROGRAM_TO_ENGLISH[a_upper]
                    trans_lower = [t.lower() for t in translations]
                    
                    if e_lower in trans_lower:
                        sumerogram_score += sumerogram_bonus
                        break
                    # Check plural forms
                    if e_lower.endswith('s') and e_lower[:-1] in trans_lower:
                        sumerogram_score += sumerogram_bonus
                        break

        # Combine all scores
        final_score = coverage_log_prob + length_penalty + ratio_penalty + boundary_score + proper_noun_score + sumerogram_score

        # Store in cache (note: cache key doesn't include eng_text, so only cache when not using PN bonus)
        if not eng_text:
            self.span_cache[cache_key] = final_score

        return final_score

### 2.3 Global Alignment with Viterbi DP

The core alignment algorithm using dynamic programming to find globally optimal sentence-level alignment.

**Design:**
- **English**: Split into sentences (punctuation-based)
- **Akkadian**: Continuous token sequence (no reliable sentence markers)
- **Output**: Optimal Akkadian token spans for each English sentence

**Algorithm:**
1. **Anchor Finding**: High-confidence alignment points (proper nouns via Soundex, gap markers)
2. **DP Table**: `dp[i][j]` = best score aligning first `i` English sentences to first `j` Akkadian tokens
3. **Diagonal Prior**: Penalizes deviation from expected position (encourages monotonic alignment)
4. **Backtracking**: Extracts optimal alignment path

In [ ]:
# Sumerogram/Logogram to English mappings
# Include both accented and normalized forms for matching
SUMEROGRAM_TO_ENGLISH = {
    # Family terms
    'DUMU': ['son', 's.', 'daughter'],
    'DUMU.MUNUS': ['daughter', 'd.'],
    'DUMU.NITA': ['son', 's.'],
    'AMA': ['mother'],
    'AD': ['father', 'f.'],
    'DAM': ['wife', 'spouse'],
    'NIN': ['sister'],
    'ŠEŠ': ['brother'],
    'SES': ['brother'],
    
    # Legal/witness terms
    'IGI': ['witness', 'witnesses', 'before', 'in presence'],
    'ŠU': ['hand', 'by hand of'],
    'SU': ['hand', 'by hand of'],
    
    # Metals/commodities
    'KÙ.BABBAR': ['silver'],
    'KU.BABBAR': ['silver'],
    'KÙ.GI': ['gold'],
    'KU.GI': ['gold'],
    'URUDU': ['copper'],
    'AN.NA': ['tin'],
    
    # Time terms
    'ITU': ['month'],
    'ITI': ['month'],
    'ITU.1.KAM': ['month'],
    'MU': ['year'],
    'UD': ['day'],
    
    # Place/institution terms
    'É': ['house', 'temple', 'estate'],
    'E': ['house', 'temple', 'estate'],
    'É.GAL': ['palace'],
    'E.GAL': ['palace'],
    'URU': ['city', 'town'],
    'KUR': ['land', 'country', 'mountain'],
    
    # Person/status terms  
    'LUGAL': ['king'],
    'MUNUS': ['woman'],
    'LÚ': ['man'],
    'LU': ['man'],
    'DINGIR': ['god', 'divine'],
    'GEME': ['slave girl', 'female slave'],
    'ÌR': ['slave', 'servant'],
    'IR': ['slave', 'servant'],
    
    # Common transaction terms
    'ŠÀM': ['price'],
    'SAM': ['price'],
    'A.ŠÀ': ['field'],
    'A.SA': ['field'],
    
    # Weights/measures
    'GÍN': ['shekel', 'shekels'],
    'GIN': ['shekel', 'shekels'],
    'GÍN.TA': ['shekels'],
    'GIN.TA': ['shekels'],
    'MA.NA': ['mina', 'minas'],
    'GUR': ['gur', 'kor'],
}

print(f"Loaded {len(SUMEROGRAM_TO_ENGLISH)} Sumerogram mappings")


def normalize_sumerogram(tok):
    """Normalize a token for Sumerogram matching."""
    import unicodedata
    normalized = unicodedata.normalize('NFD', tok)
    normalized = ''.join(c for c in normalized if unicodedata.category(c) != 'Mn')
    return normalized.upper()


# Load lexicon types for proper noun detection
# PN = Proper Noun, DN = Divine Name, GN = Geographic Name
LEXICON_TYPES = load_lexicon_types()
PROPER_NOUN_TYPES = {'PN', 'DN', 'GN', 'RN', 'MN', 'TN', 'WN'}  # All name types
print(f"Loaded {len(LEXICON_TYPES)} lexicon type annotations")
print(f"  PN (proper nouns): {sum(1 for t in LEXICON_TYPES.values() if t == 'PN')}")
print(f"  DN (divine names): {sum(1 for t in LEXICON_TYPES.values() if t == 'DN')}")
print(f"  GN (geographic names): {sum(1 for t in LEXICON_TYPES.values() if t == 'GN')}")


def is_proper_noun_in_lexicon(akk_token):
    """Check if an Akkadian token is a proper noun according to the lexicon."""
    if akk_token in LEXICON_TYPES:
        return LEXICON_TYPES[akk_token] in PROPER_NOUN_TYPES
    norm_token = normalize_akkadian_string(akk_token)
    if norm_token in LEXICON_TYPES:
        return LEXICON_TYPES[norm_token] in PROPER_NOUN_TYPES
    dehyphenated = akk_token.replace('-', '')
    if dehyphenated in LEXICON_TYPES:
        return LEXICON_TYPES[dehyphenated] in PROPER_NOUN_TYPES
    return False


def tokenize_english_preserve_case(text):
    """English tokenization that PRESERVES case for proper noun detection."""
    if not text: return []
    text = re.sub(r'[()\[\]]', '', text)
    tokens = [t for t in re.findall(r'<big_gap>|<gap>|\w+(?:[\.\-]\w+)*', text, re.UNICODE) if t]
    filtered_tokens = [t for t in tokens if t not in ['-a', '-i', '-ma', '-u', '-am']]
    return filtered_tokens


def normalize_for_soundex(text):
    """Normalize text for Soundex matching - handles Akkadian special characters."""
    text = text.replace('-', '')
    replacements = {
        'š': 's', 'Š': 'S', 'ṣ': 's', 'Ṣ': 'S', 'ṭ': 't', 'Ṭ': 'T',
        'ḫ': 'h', 'Ḫ': 'H', 'ŋ': 'n', 'Ŋ': 'N', 'ʾ': '', 'ʿ': '',
        '₀': '0', '₁': '1', '₂': '2', '₃': '3', '₄': '4',
        '₅': '5', '₆': '6', '₇': '7', '₈': '8', '₉': '9',
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    text = normalize_akkadian_string(text)
    return text


def extract_proper_nouns_from_text(text):
    """
    Extract potential proper nouns from English text.
    Proper nouns are capitalized words (not at sentence start) that aren't stopwords.
    """
    words = re.findall(r'\b[A-Za-zÀ-ÿ][\w-]*\b', text)
    proper_nouns = []
    prev_was_sentence_end = True
    
    for i, word in enumerate(words):
        if i == 0:
            prev_was_sentence_end = False
            continue
        if i > 0:
            word_pos = text.find(word, text.find(words[i-1]) + len(words[i-1]) if i > 0 else 0)
            if word_pos > 0:
                before = text[:word_pos]
                if re.search(r'[.!?]\s*$', before):
                    prev_was_sentence_end = True
        if prev_was_sentence_end:
            prev_was_sentence_end = False
            continue
        if not word[0].isupper():
            continue
        if word.lower() in STOPWORDS:
            continue
        if len(word) < 3:
            continue
        proper_nouns.append(word)
        prev_was_sentence_end = False
    return proper_nouns


def find_token_anchors(eng_sentences, akk_tokens, verbose=False):
    """Find anchor points between English sentences and Akkadian tokens."""
    anchors = []
    eng_word_occurrences = {}
    for sent_idx, sent in enumerate(eng_sentences):
        words = tokenize_english(sent)
        for word_pos, word in enumerate(words):
            key = word.lower()
            if key not in eng_word_occurrences:
                eng_word_occurrences[key] = []
            eng_word_occurrences[key].append((sent_idx, word_pos))
    
    sumerogram_occurrences = {}
    for akk_idx, akk_tok in enumerate(akk_tokens):
        akk_upper = akk_tok.upper()
        akk_normalized = normalize_sumerogram(akk_tok)
        for form in [akk_upper, akk_normalized]:
            if form in SUMEROGRAM_TO_ENGLISH:
                if form not in sumerogram_occurrences:
                    sumerogram_occurrences[form] = []
                sumerogram_occurrences[form].append(akk_idx)
                break
    
    used_akk_indices = set()
    used_eng_occs = set()
    all_sumerogram_matches = []
    
    for sumerogram, akk_indices in sumerogram_occurrences.items():
        eng_translations = SUMEROGRAM_TO_ENGLISH[sumerogram]
        eng_occs = []
        for eng_trans in eng_translations:
            eng_key = eng_trans.lower()
            if eng_key in eng_word_occurrences:
                eng_occs.extend(eng_word_occurrences[eng_key])
            if eng_key.endswith('s') and eng_key[:-1] in eng_word_occurrences:
                eng_occs.extend(eng_word_occurrences[eng_key[:-1]])
            elif not eng_key.endswith('s') and (eng_key + 's') in eng_word_occurrences:
                eng_occs.extend(eng_word_occurrences[eng_key + 's'])
        eng_occs = sorted(set(eng_occs))
        for akk_idx in akk_indices:
            all_sumerogram_matches.append((akk_idx, sumerogram, eng_occs, eng_translations))
    
    all_sumerogram_matches.sort(key=lambda x: x[0])
    
    for akk_idx, sumerogram, eng_occs, eng_translations in all_sumerogram_matches:
        if akk_idx in used_akk_indices:
            continue
        best_eng = None
        for eng_sent_idx, eng_word_pos in eng_occs:
            if (eng_sent_idx, eng_word_pos) not in used_eng_occs:
                best_eng = (eng_sent_idx, eng_word_pos)
                break
        if best_eng:
            eng_sent_idx, eng_word_pos = best_eng
            anchors.append({
                'eng_idx': eng_sent_idx,
                'akk_idx': akk_idx,
                'type': 'sumerogram',
                'word': f"{akk_tokens[akk_idx]}->{eng_translations[0]}"
            })
            used_akk_indices.add(akk_idx)
            used_eng_occs.add(best_eng)
    
    # Proper noun matching using lexicon
    akk_proper_nouns = {}
    for akk_idx, akk_tok in enumerate(akk_tokens):
        if akk_idx in used_akk_indices:
            continue
        if is_proper_noun_in_lexicon(akk_tok):
            norm = normalize_akkadian_string(akk_tok)
            if norm not in akk_proper_nouns:
                akk_proper_nouns[norm] = []
            akk_proper_nouns[norm].append(akk_idx)
    
    eng_proper_nouns = {}
    for sent_idx, sent in enumerate(eng_sentences):
        words = tokenize_english_preserve_case(sent)
        for word in words:
            if word[0].isupper() and word.lower() not in STOPWORDS:
                norm = word.lower()
                if norm not in eng_proper_nouns:
                    eng_proper_nouns[norm] = []
                eng_proper_nouns[norm].append(sent_idx)
    
    for eng_word, eng_sents_with_word in eng_proper_nouns.items():
        eng_normalized = normalize_for_soundex(eng_word)
        eng_soundex = soundex(eng_normalized)
        if not eng_soundex:
            continue
        for akk_word, akk_indices in akk_proper_nouns.items():
            akk_letters = ''.join(c for c in akk_word if c.isalpha())
            if any(c.isupper() for c in akk_letters):
                continue
            akk_normalized = normalize_for_soundex(akk_word)
            akk_soundex = soundex(akk_normalized)
            if akk_soundex and eng_soundex == akk_soundex:
                eng_sents_with_name = sorted(set(eng_sents_with_word))
                available_akk = sorted([idx for idx in akk_indices if idx not in used_akk_indices])
                n_matches = min(len(eng_sents_with_name), len(available_akk))
                for i in range(n_matches):
                    eng_sent_idx = eng_sents_with_name[i]
                    akk_idx = available_akk[i]
                    anchors.append({
                        'eng_idx': eng_sent_idx,
                        'akk_idx': akk_idx,
                        'type': 'proper_noun',
                        'word': f"{eng_word}<->{akk_word}"
                    })
                    used_akk_indices.add(akk_idx)
    
    # Soundex fallback for remaining proper nouns
    for sent_idx, sent in enumerate(eng_sentences):
        words = tokenize_english_preserve_case(sent)
        for eng_word in words:
            if not eng_word[0].isupper() or eng_word.lower() in STOPWORDS:
                continue
            if len(eng_word) < 3:
                continue
            eng_normalized = normalize_for_soundex(eng_word)
            eng_soundex = soundex(eng_normalized)
            if not eng_soundex:
                continue
            for akk_idx, akk_tok in enumerate(akk_tokens):
                if akk_idx in used_akk_indices:
                    continue
                akk_letters = ''.join(c for c in akk_tok if c.isalpha())
                if any(c.isupper() for c in akk_letters):
                    continue
                akk_normalized = normalize_for_soundex(akk_tok)
                akk_soundex = soundex(akk_normalized)
                if akk_soundex and eng_soundex == akk_soundex:
                    anchors.append({
                        'eng_idx': sent_idx,
                        'akk_idx': akk_idx,
                        'type': 'proper_noun_heuristic',
                        'word': f"{eng_word}<->{akk_tok}"
                    })
                    used_akk_indices.add(akk_idx)
                    break
    
    anchors.sort(key=lambda x: (x['eng_idx'], x['akk_idx']))
    filtered = []
    last_akk = -1
    for anchor in anchors:
        if anchor['akk_idx'] > last_akk:
            filtered.append(anchor)
            last_akk = anchor['akk_idx']
    
    if verbose:
        print(f"Found {len(filtered)} anchors (bijection matching)")
        by_type = {}
        for a in filtered:
            by_type[a['type']] = by_type.get(a['type'], 0) + 1
        print(f"  By type: {by_type}")
    
    return filtered


ENGLISH_CLAUSE_STARTERS = {'if', 'when', 'whoever', 'should', 'whereas', 'whenever', 'unless'}


def compute_ideal_scores(eng_sentences, akk_tokens, scorer, akk_props=None):
    """
    SCOUT PHASE: For each English sentence, find its best possible score
    if it could take ANY span from the document (ignoring other sentences).
    
    This is used to normalize scores so each sentence is judged relative
    to its own potential, preventing "easy" sentences from dominating "hard" ones.
    
    Args:
        eng_sentences: List of English sentences
        akk_tokens: List of Akkadian tokens
        scorer: AlignmentScorer instance
        akk_props: Optional precomputed Akkadian properties
        
    Returns:
        List of (ideal_score, ideal_span) for each sentence
    """
    n_akk = len(akk_tokens)
    if akk_props is None:
        akk_props = scorer.precompute_akkadian_props(akk_tokens)
    
    ratio_upper = math.exp(scorer.mu_log + 2.5 * scorer.sigma_log)
    ratio_lower = math.exp(scorer.mu_log - 2.5 * scorer.sigma_log)
    
    ideal_scores = []
    
    for sent_idx, sent in enumerate(eng_sentences):
        eng_toks = tokenize_english(sent)
        n_eng_toks = len(eng_toks)
        
        if n_eng_toks == 0:
            ideal_scores.append((-100.0, None))
            continue
        
        # Search all possible spans for this sentence
        best_score = float('-inf')
        best_span = None
        
        max_span = min(int(n_eng_toks * ratio_upper) + 10, n_akk)
        min_span = max(1, int(n_eng_toks * ratio_lower) - 5)
        
        for start in range(n_akk):
            for length in range(min_span, max_span + 1):
                end = start + length
                if end > n_akk:
                    break
                
                span_props = akk_props[start:end]
                score = scorer.score_span(eng_toks, akk_tokens[start:end], span_props, eng_text=sent)
                
                if score > best_score:
                    best_score = score
                    best_span = (start, end)
        
        ideal_scores.append((best_score, best_span))
    
    return ideal_scores


def global_align(eng_sentences, akk_tokens, scorer,
                 allow_partial=False, lookahead=1, diagonal_weight=0.3,
                 anchor_bonus=5.0, clause_penalty=-5.0,
                 verbose=False):
    """
    Find optimal global alignment between English SENTENCES and Akkadian TOKEN SPANS.
    
    Uses NORMALIZED REGRET scoring: each sentence's score is normalized by its
    "ideal" (best possible) score. This prevents easy sentences from dominating
    hard sentences and reduces NULL alignments.
    
    KEY FEATURES:
    - Scout phase computes ideal score for each sentence
    - DP uses satisfaction ratios (actual/ideal) instead of raw scores
    - This protects hard-to-align sentences from being abandoned
    - Anchors (proper nouns, Sumerograms) are soft constraints with bonuses
    - Clause-starter penalty discourages merging sentences starting with "If", etc.
    
    Args:
        eng_sentences: List of English sentences
        akk_tokens: List of Akkadian tokens (flat)
        scorer: AlignmentScorer instance
        allow_partial: If True, can skip leading Akkadian tokens
        lookahead: Max sentences to merge (1 = no merging)
        diagonal_weight: Penalty weight for deviating from diagonal
        anchor_bonus: Bonus for including anchor tokens in span
        clause_penalty: Penalty for merging sentences with clause starters
        verbose: Print debug info
        
    Returns:
        List of alignment dicts with 'eng_range', 'akk_range', 'english', 
        'akkadian', 'score', 'is_null' keys
    """
    n_eng = len(eng_sentences)
    n_akk = len(akk_tokens)
    
    if n_eng == 0 or n_akk == 0:
        return []
    
    # Precompute Akkadian properties once
    akk_props = scorer.precompute_akkadian_props(akk_tokens)
    
    # SCOUT PHASE: Find ideal scores for normalized regret
    ideal_scores = compute_ideal_scores(eng_sentences, akk_tokens, scorer, akk_props)
    
    if verbose:
        print("\\n🔍 SCOUT PHASE: Ideal scores for each sentence:")
        for i, (score, span) in enumerate(ideal_scores):
            print(f"  Sent {i}: ideal_score={score:.2f}, ideal_span={span}")
    
    # Find anchors using proportional assignment
    anchors_raw = find_token_anchors(eng_sentences, akk_tokens, verbose=verbose)
    constraints = [[] for _ in range(n_eng)]
    for anchor in anchors_raw:
        constraints[anchor['eng_idx']].append(anchor['akk_idx'])
    
    ratio_upper = math.exp(scorer.mu_log + 2.5 * scorer.sigma_log)
    ratio_lower = math.exp(scorer.mu_log - 2.5 * scorer.sigma_log)
    
    INF = float('-inf')
    dp = [[None for _ in range(n_akk + 1)] for _ in range(n_eng + 1)]
    
    if allow_partial:
        for j in range(n_akk + 1):
            dp[0][j] = (0.0, None)
    else:
        dp[0][0] = (0.0, None)
    
    # ALIGN PHASE: DP with normalized satisfaction scores
    for i in range(n_eng + 1):
        for j in range(n_akk + 1):
            if dp[i][j] is None:
                continue
            
            current_score, _ = dp[i][j]
            if current_score == INF:
                continue
            
            for eng_size in range(1, min(lookahead + 1, n_eng - i + 1)):
                new_i = i + eng_size
                
                merged_eng = ' '.join(eng_sentences[i:new_i])
                eng_tokens_list = tokenize_english(merged_eng)
                n_eng_tokens = len(eng_tokens_list)
                
                # Clause-starter and merge penalties (normalized scale)
                clause_starter_penalty = 0.0
                merge_penalty = 0.0
                if eng_size > 1:
                    merge_penalty = -0.1 * (eng_size - 1)
                    for s_idx in range(i + 1, new_i):
                        sent = eng_sentences[s_idx]
                        first_word = sent.split()[0].lower() if sent.split() else ''
                        if first_word in ENGLISH_CLAUSE_STARTERS:
                            clause_starter_penalty += -0.05
                
                sentence_anchors = []
                for s_idx in range(i, new_i):
                    sentence_anchors.extend(constraints[s_idx])
                
                has_anchors = len(sentence_anchors) > 0
                
                # Get ideal score for normalization (use first sentence in merge)
                ideal_score, _ = ideal_scores[i]
                normalization_factor = abs(ideal_score) if ideal_score != 0 else 1.0
                
                # Option A: NULL Alignment
                # NULL gets a fixed "satisfaction" score worse than achieving potential
                null_raw = -100 - 0.5 * n_eng_tokens
                null_satisfaction = null_raw / normalization_factor
                
                if has_anchors:
                    null_satisfaction -= 0.2 * len(sentence_anchors)
                
                new_j_null = j
                total_null = current_score + null_satisfaction + merge_penalty + clause_starter_penalty
                
                if dp[new_i][new_j_null] is None or total_null > dp[new_i][new_j_null][0]:
                    dp[new_i][new_j_null] = (total_null, (i, j, eng_size, 0))
                
                # Option B: Normal Alignment with NORMALIZED scoring
                if n_eng_tokens == 0:
                    continue
                
                max_akk_span = int(n_eng_tokens * ratio_upper) + 10
                min_akk_span = max(1, int(n_eng_tokens * ratio_lower) - 5)
                
                for new_j in range(j + min_akk_span, min(j + max_akk_span + 1, n_akk + 1)):
                    akk_span_len = new_j - j
                    akk_span_tokens = akk_tokens[j:new_j]
                    akk_span_props = akk_props[j:new_j]
                    
                    # Get raw score and normalize by ideal (satisfaction ratio)
                    raw_score = scorer.score_span(eng_tokens_list, akk_span_tokens, akk_span_props,
                                                   eng_text=merged_eng)
                    satisfaction = raw_score / normalization_factor
                    
                    # Diagonal penalty (normalized scale)
                    expected_j = (new_i / n_eng) * n_akk if n_eng > 0 else new_j
                    diag_penalty = -0.01 * abs(new_j - expected_j)
                    
                    # Anchor bonus (normalized scale)
                    anchor_score = 0.0
                    for akk_idx in sentence_anchors:
                        if j <= akk_idx < new_j:
                            anchor_score += 0.1
                        else:
                            anchor_score -= 0.05
                    
                    total_score = current_score + satisfaction + diag_penalty + anchor_score + clause_starter_penalty + merge_penalty
                    
                    if dp[new_i][new_j] is None or total_score > dp[new_i][new_j][0]:
                        dp[new_i][new_j] = (total_score, (i, j, eng_size, akk_span_len))
    
    # Find best ending state
    best_score = INF
    best_end = None
    
    for j in range(n_akk + 1):
        if dp[n_eng][j] is not None and dp[n_eng][j][0] > best_score:
            best_score = dp[n_eng][j][0]
            best_end = (n_eng, j)
    
    if best_end is None:
        if verbose:
            print("Warning: No valid alignment path found")
        return []
    
    # Backtrack to build alignments
    alignments = []
    i, j = best_end
    
    while i > 0:
        if dp[i][j] is None or dp[i][j][1] is None:
            break
        
        prev_i, prev_j, eng_size, akk_size = dp[i][j][1]
        merged_eng = ' '.join(eng_sentences[prev_i:i])
        
        if akk_size == 0:
            merged_akk = ''
            score = -100.0
            is_null = True
        else:
            merged_akk = ' '.join(akk_tokens[prev_j:j])
            eng_toks = tokenize_english(merged_eng)
            akk_span_props = akk_props[prev_j:j]
            score = scorer.score_span(eng_toks, akk_tokens[prev_j:j], akk_span_props, eng_text=merged_eng)
            is_null = False
        
        alignments.append({
            'eng_range': (prev_i, i),
            'akk_range': (prev_j, j),
            'english': merged_eng,
            'akkadian': merged_akk,
            'score': score,
            'is_null': is_null
        })
        
        i, j = prev_i, prev_j
    
    alignments.reverse()
    
    if verbose:
        null_count = sum(1 for a in alignments if a.get('is_null', False))
        print(f"Viterbi produced {len(alignments)} alignment pairs ({null_count} NULL alignments)")
    
    return alignments


print("Global alignment with NORMALIZED REGRET scoring defined.")
print("  Key improvement: Each sentence judged relative to its potential")
print("  - Scout phase finds ideal score for each sentence")
print("  - DP uses satisfaction = actual_score / |ideal_score|")
print("  - Prevents easy sentences from abandoning hard ones")
print(f"  - Clause starters: {ENGLISH_CLAUSE_STARTERS}")

### 2.4 Training Set Generation Pipeline

Complete pipeline for generating aligned training data:
1. Split texts into sentences
2. Apply global alignment with quality filtering
3. Extract partial alignments from noisy documents
4. Output clean training pairs (JSONL format)

In [ ]:
def split_sentences(text, is_akkadian=False):
    """
    Split text into sentences for alignment.
    Handles both English and Akkadian text.
    """
    if is_akkadian:
        # Akkadian: Split on newlines (lines are sentences)
        lines = [l.strip() for l in text.split('\n') if l.strip()]
        return lines
    else:
        # English: Handle abbreviations before splitting
        # Protect common abbreviations from being treated as sentence endings
        protected = text
        
        # Common abbreviations in Akkadian studies: s. (son of), d. (daughter of), f. (father)
        # Pattern: single letter followed by period before a space and capital/name
        # Replace "s. Name" patterns temporarily
        abbrev_pattern = r'\b([sdfSDF])\.\s+(?=[A-ZḪŠṢṬ])'
        protected = re.sub(abbrev_pattern, r'\1__DOT__ ', protected)
        
        # Also protect other common abbreviations
        for abbr in ['Mr.', 'Mrs.', 'Dr.', 'vs.', 'etc.', 'i.e.', 'e.g.', 'no.', 'No.']:
            protected = protected.replace(abbr, abbr.replace('.', '__DOT__'))
        
        # Split on sentence-ending punctuation
        sentences = []
        parts = re.split(r'(?<=[.!?])\s+', protected)
        for part in parts:
            # Further split on newlines
            for line in part.split('\n'):
                line = line.strip()
                if line:
                    # Restore periods in abbreviations
                    line = line.replace('__DOT__', '.')
                    sentences.append(line)
        return sentences


def quality_filter(alignment, min_score=-50.0, max_ratio=3.0, min_coverage=0.2):
    """
    Filter out low-quality alignments.
    
    Args:
        alignment: Dict with 'english', 'akkadian', 'score' keys
        min_score: Minimum alignment score
        max_ratio: Maximum length ratio (words)
        min_coverage: Minimum coverage ratio
        
    Returns:
        True if alignment passes quality filters
    """
    eng_tokens = alignment['english'].split()
    akk_tokens = alignment['akkadian'].split()
    
    n_eng = len(eng_tokens)
    n_akk = len(akk_tokens)
    
    if n_eng == 0 or n_akk == 0:
        return False
    
    # Score filter
    if alignment.get('score', 0) < min_score:
        return False
    
    # Length ratio filter
    ratio = n_akk / n_eng
    if ratio > max_ratio or ratio < 1/max_ratio:
        return False
    
    # Numeric mismatch filter
    eng_nums = set(re.findall(r'\d+', alignment['english']))
    akk_nums = set(re.findall(r'\d+', alignment['akkadian']))
    
    # Allow if both have numbers or both don't
    if bool(eng_nums) != bool(akk_nums):
        return False
    
    return True


def extract_partial_alignments(eng_sentences, akk_tokens, scorer, 
                               window_size=5, min_score=-30.0):
    """
    Extract partial alignments from noisy documents.
    Slides a window to find local high-confidence alignments.
    
    Args:
        eng_sentences: List of English sentences
        akk_tokens: List of Akkadian tokens (flat list)
        scorer: AlignmentScorer instance
        window_size: Size of sliding window (English sentences)
        min_score: Minimum score threshold
        
    Returns:
        List of partial alignment dicts
    """
    partials = []
    
    n_eng = len(eng_sentences)
    n_akk = len(akk_tokens)
    
    if n_eng == 0 or n_akk == 0:
        return partials
    
    # Estimate rough token ratio
    avg_eng_tokens = sum(len(s.split()) for s in eng_sentences) / n_eng
    tokens_per_sent = n_akk / n_eng if n_eng > 0 else 10
    
    # Slide window along English sentences
    for e_start in range(0, n_eng - window_size + 1, window_size // 2):
        e_end = min(e_start + window_size, n_eng)
        
        # Estimate corresponding Akkadian token range
        a_center = int(e_start * tokens_per_sent)
        a_start = max(0, a_center - int(window_size * tokens_per_sent))
        a_end = min(n_akk, a_center + int(window_size * tokens_per_sent * 2))
        
        if a_start >= a_end:
            continue
        
        # Run alignment on this window
        eng_window = eng_sentences[e_start:e_end]
        akk_window = akk_tokens[a_start:a_end]
        
        try:
            local_alignments = global_align(
                eng_window, akk_window, scorer,
                allow_partial=True,
                lookahead=2,
                verbose=False
            )
        except Exception as e:
            continue
        
        # Filter and add good alignments
        for aln in local_alignments:
            if quality_filter(aln, min_score=min_score):
                # Adjust indices to global positions
                aln['global_eng_range'] = (
                    e_start + aln['eng_range'][0],
                    e_start + aln['eng_range'][1]
                )
                aln['global_akk_range'] = (
                    a_start + aln['akk_range'][0],
                    a_start + aln['akk_range'][1]
                )
                partials.append(aln)
    
    # Deduplicate overlapping alignments
    partials.sort(key=lambda x: (-x['score'], x['global_eng_range'][0]))
    used_eng = set()
    used_akk = set()
    deduped = []
    
    for p in partials:
        eng_range = set(range(p['global_eng_range'][0], p['global_eng_range'][1]))
        akk_range = set(range(p['global_akk_range'][0], p['global_akk_range'][1]))
        
        if not (eng_range & used_eng) and not (akk_range & used_akk):
            deduped.append(p)
            used_eng.update(eng_range)
            used_akk.update(akk_range)
    
    return deduped


def generate_training_set(documents, scorer, output_path, 
                          use_partial=True, verbose=True):
    """
    Generate a clean training set from parallel documents.
    
    Args:
        documents: List of dicts with 'english' and 'akkadian' keys
        scorer: AlignmentScorer instance
        output_path: Path to output JSONL file
        use_partial: Whether to extract partial alignments for noisy docs
        verbose: Print progress
        
    Returns:
        Statistics dict
    """
    stats = {
        'total_docs': len(documents),
        'aligned_pairs': 0,
        'partial_pairs': 0,
        'failed_docs': 0
    }
    
    with open(output_path, 'w', encoding='utf-8') as f:
        for doc_idx, doc in enumerate(documents):
            if verbose and doc_idx % 100 == 0:
                print(f"Processing document {doc_idx}/{len(documents)}...")
            
            eng_text = doc.get('english', '')
            akk_text = doc.get('akkadian', '')
            
            if not eng_text or not akk_text:
                stats['failed_docs'] += 1
                continue
            
            # Split English into sentences
            eng_sentences = split_sentences(eng_text, is_akkadian=False)
            
            # Tokenize Akkadian as flat token sequence
            akk_text_clean = re.sub(r'\s+', ' ', akk_text).strip()
            akk_tokens = tokenize_akkadian(akk_text_clean)
            
            if not eng_sentences or not akk_tokens:
                stats['failed_docs'] += 1
                continue
            
            try:
                # Try global alignment first
                alignments = global_align(
                    eng_sentences, akk_tokens, scorer,
                    allow_partial=False,
                    verbose=False
                )
                
                # Check overall quality
                good_alignments = [a for a in alignments if quality_filter(a)]
                coverage = len(good_alignments) / len(alignments) if alignments else 0
                
                if coverage > 0.5:
                    # Good document - use full alignments
                    for aln in good_alignments:
                        output = {
                            'english': aln['english'],
                            'akkadian': aln['akkadian'],
                            'score': aln['score'],
                            'doc_id': doc.get('id', doc_idx),
                            'type': 'full'
                        }
                        f.write(json.dumps(output, ensure_ascii=False) + '\n')
                        stats['aligned_pairs'] += 1
                
                elif use_partial:
                    # Noisy document - extract partial alignments
                    partials = extract_partial_alignments(
                        eng_sentences, akk_tokens, scorer
                    )
                    for aln in partials:
                        output = {
                            'english': aln['english'],
                            'akkadian': aln['akkadian'],
                            'score': aln['score'],
                            'doc_id': doc.get('id', doc_idx),
                            'type': 'partial'
                        }
                        f.write(json.dumps(output, ensure_ascii=False) + '\n')
                        stats['partial_pairs'] += 1
                        
            except Exception as e:
                stats['failed_docs'] += 1
                if verbose:
                    print(f"  Error processing doc {doc_idx}: {e}")
    
    if verbose:
        print(f"\nGeneration complete:")
        print(f"  Total documents: {stats['total_docs']}")
        print(f"  Aligned pairs: {stats['aligned_pairs']}")
        print(f"  Partial pairs: {stats['partial_pairs']}")
        print(f"  Failed docs: {stats['failed_docs']}")
    
    return stats


print("Training set generation pipeline defined.")

### 2.5 End-to-End Demo

Complete demonstration: create scorer from pre-trained models and run Viterbi alignment on sample documents.

In [ ]:
# =============================================================================
# Test Alignment on Sample Documents from train.csv (Full Output)
# =============================================================================
import random
import pandas as pd
from collections import Counter

print("=" * 80)
print("ALIGNMENT TEST: Documents from train.csv")
print("=" * 80)

# Load documents directly from train.csv (document-level parallel text)
train_df = pd.read_csv('train.csv')
print(f"Loaded {len(train_df)} documents from train.csv")

# Get pre-trained models from results
lexical_model = results['lexical_model']
form_to_lexeme = results['form_to_lexeme']
length_model = results['length_model']

# Compute IDF counts from train.csv documents
idf_counts = Counter()
for idx, row in train_df.iterrows():
    eng_text = row['translation']
    if pd.notna(eng_text):
        doc_words = set(tokenize_english(eng_text))
        for w in doc_words:
            idf_counts[w] += 1

# Create scorer
scorer = AlignmentScorer(
    lexical_model=lexical_model,
    form_to_lexeme=form_to_lexeme,
    idf_counts=idf_counts,
    total_docs=len(train_df),
    mu_log=length_model.get('mu_log', 0.0),
    sigma_log=length_model.get('sigma_log', 1.0)
)

# Sample 10 random documents
random.seed(42)
sample_indices = random.sample(range(len(train_df)), min(10, len(train_df)))

total_alns = 0
total_nulls = 0
total_merges = 0
all_scores = []

for doc_idx, row_idx in enumerate(sample_indices):
    row = train_df.iloc[row_idx]
    doc_id = row['oare_id']
    
    # Get text and apply replace_gaps for proper gap marker handling
    akk_text = replace_gaps(row['transliteration']) if pd.notna(row['transliteration']) else ''
    eng_text = replace_gaps(row['translation']) if pd.notna(row['translation']) else ''
    
    if not akk_text or not eng_text:
        continue
    
    # Tokenize Akkadian and split English into sentences
    doc_akk_tokens = tokenize_akkadian(akk_text)
    doc_eng_sents = split_sentences(eng_text, is_akkadian=False)
    
    if len(doc_eng_sents) < 2 or len(doc_akk_tokens) < 5:
        continue
    
    # Run alignment
    alignments = global_align(doc_eng_sents, doc_akk_tokens, scorer, verbose=False)
    
    # Count stats
    null_count = sum(1 for a in alignments if a.get('is_null', False))
    merge_count = sum(1 for a in alignments if a['eng_range'][1] - a['eng_range'][0] > 1)
    scores = [a['score'] for a in alignments]
    
    total_alns += len(alignments)
    total_nulls += null_count
    total_merges += merge_count
    all_scores.extend(scores)
    
    # Show full alignment details
    print(f"\n{'='*80}")
    print(f"Doc {doc_idx+1}: {doc_id}")
    print(f"  Sentences: {len(doc_eng_sents)}, Tokens: {len(doc_akk_tokens)}")
    print("-" * 80)
    for i, a in enumerate(alignments):
        eng_range = a['eng_range']
        akk_range = a['akk_range']
        is_null = a.get('is_null', False)
        eng_text = a['english']
        akk_text = a['akkadian']
        
        status = "[NULL]" if is_null else f"[Akk {akk_range[0]}:{akk_range[1]}]"
        print(f"\n  {i+1}. Eng[{eng_range[0]}:{eng_range[1]}] → {status}")
        print(f"     Eng: {eng_text}")
        if not is_null:
            print(f"     Akk: {akk_text}")
        print(f"     Score: {a['score']:.2f}")
    
    print(f"\n  Doc Summary: {len(alignments)} alignments, {null_count} NULLs, {merge_count} merges")

print("\n" + "=" * 80)
print("OVERALL SUMMARY")
print("=" * 80)
print(f"Total alignments: {total_alns}")
print(f"NULL alignments: {total_nulls} ({100*total_nulls/max(total_alns,1):.1f}%)")
print(f"Merged alignments: {total_merges} ({100*total_merges/max(total_alns,1):.1f}%)")
print(f"Average score: {sum(all_scores)/max(len(all_scores),1):.2f}")

## 🏁 Conclusion: Decoding the Deep Past

This notebook demonstrates an end-to-end unsupervised pipeline for aligning complex, low-resource Akkadian texts. By innovating with **Normalized Regret Scoring** and strictly enforcing **Monotonic Anchoring**, we overcame the density and fragmentation issues that plague standard alignment algorithms. This pipeline provides a foundation for generating the high-quality parallel corpora needed to train modern Neural Machine Translation models for the ancient world.

### ⚠️ Limitations
While effective, this approach has constraints:
*   **Strict Monotonicity:** The assumption that translation order mimics source order is generally true for these texts but fails on complex poetic structures or heavy reordering.
*   **Lexicon Dependency:** Our anchoring relies heavily on existing dictionaries. Unknown proper nouns (OOV) can lead to drift in the alignment path.
*   **Fragment Sensitivity:** Extremely short fragments (1-2 signs) still pose a significant challenge for statistical confidence.
*   **Noisy Translations:** The dataset is not perfectly parallel. Some English translations contain interpolations, commentary, or inferred context that do not exist in the Akkadian source. These "hallucinated" segments confuse the model, which attempts to force an alignment where none exists.
*   **Asymmetric Gaps:** Handling lacunae (missing text) is difficult when the damage is inconsistently represented—e.g., marked as `[...]` in Akkadian but smoothed over or guessed in the English translation.

### 🏛️ The Deep Past Challenge
This work directly supports the **Deep Past Challenge**, which asks a bold question: *Can AI decode 4,000-year-old business records?*

The dataset used here represents the everyday business records of ancient Assyrian merchants. With over **8,000 cuneiform texts** available and thousands more lying unread in museum drawers, the potential for discovery is immense. We are not just training models; we are bringing ancient voices back into the story of humanity.

### 🔮 Next Steps & Call for Collaboration 🤝
I am actively looking for teammates to join me in the **Deep Past Challenge**. While this notebook establishes a strong baseline, we can go further together:

*   **Scale Up:** Applying this pipeline to the full 8,000-text competition corpus.
*   **Improve Anchors:** Integrating better NER for Old Assyrian names to reduce alignment drift.
*   **Hybrid Models:** Combining this statistical backbone with modern embeddings for better semantic matching.

If you are an NLP researcher, Assyriologist, or data scientist interested in **collaborating on this competition entry**, please reach out! Let's team up to win this challenge and bring these ancient voices back to life.